# Layers of defence for an agent

A real kagent incident-response agent calls Claude through agentgateway and uses
operations tools deployed by kagent. It can investigate incidents, close them all,
or delete their investigation history. Tool calls really change a disposable dataset.
Each chapter adds a control and checks what changed.
Agent turns use A2A and show the returned tool-call trace. Direct HTTP and MCP requests
are labelled as boundary checks and run from the kagent agent's managed runtime.
There is no stand-in agent pod, echo model or hand-deployed tool server.

This lab uses Enterprise kagent, Enterprise agentgateway and Istio on mesh1.
Model calls use the installed Anthropic credential and are billable. Prompts use
specimen data, responses are capped and request loops are bounded.
Kindnet does not enforce NetworkPolicy, so the egress chapter controls example.com
through a namespace-scoped Istio waypoint. It does not claim a blanket internet boundary.

## Connect

Set the environment when following the notebook directly. The console sets it for every step.

In [ ]:
export DD=demo-scripts/defence DD_URL=http://dd-gateway.dd-gateway.svc
export DD_STATE="${TMPDIR:-/tmp}/defence-lab"
. "$DD/helpers.sh"

## 1. Create an incident-response agent in kagent

Build an incident-response agent in kagent and connect it to three open incidents. First, show that it can close every incident without fixing the faults. Then add policies that let it investigate but prevent those changes.

kagent creates and manages the runtime. The model is Claude, reached through agentgateway. The provider credential stays at the gateway; the agent gets its own lab identity.

<svg id="defence-architecture" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 900" width="100%" role="img" aria-labelledby="defence-title defence-desc">
<title id="defence-title">Layers of defence for a kagent incident-response agent</title>
<desc id="defence-desc">kagent runs the agent and incident MCP tools. Agentgateway enforces caller identity, tool permissions, prompt guards and rate limits. Istio refuses direct tool access and example.com egress. Before the tool policy, three unresolved incidents are closed. After the policy, the bulk close is refused and investigation remains available.</desc>
<defs>
      <linearGradient id="defence-gateway-fill" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#382763"/><stop offset="1" stop-color="#231c40"/></linearGradient>
      <filter id="defence-shadow" x="-15%" y="-15%" width="130%" height="140%"><feDropShadow dx="0" dy="5" stdDeviation="6" flood-color="#232142" flood-opacity=".09"/></filter>
      <marker id="defence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M1 1 L9 5 L1 9 Z" fill="#776b9f"/></marker>
    </defs>
<rect x="0" y="0" width="960" height="900" rx="20" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<rect x="0" y="0" width="960" height="120" rx="20" fill="#101426" stroke="none" stroke-width="1.3"/>
<rect x="0" y="90" width="960" height="30" rx="0" fill="#101426" stroke="none" stroke-width="1.3"/>
<text x="32" y="34" font-family="Arial, sans-serif" font-size="13" font-weight="700" fill="#c4b5fd" text-anchor="start" letter-spacing="2">SOLO / INCIDENT RESPONSE</text>
<text x="32" y="74" font-family="Arial, sans-serif" font-size="36" font-weight="700" fill="#ffffff" text-anchor="start">Layers of defence</text>
<text x="32" y="103" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#cbd5e1" text-anchor="start">Where the agent&#x27;s permissions are enforced</text>
<path d="M808,30 L794,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M808,30 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M822,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="808" cy="30" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="794" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="822" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="808" cy="58" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<text x="838" y="51" font-family="Arial, sans-serif" font-size="19" font-weight="700" fill="#ffffff" text-anchor="start" letter-spacing="1">SOLO.IO</text>
<rect x="842" y="76" width="86" height="28" rx="7" fill="#272c46" stroke="none" stroke-width="1.3"/>
<text x="885.0" y="95" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#d8ddec" text-anchor="middle">mesh1</text>
<g data-node="request">
<rect x="32" y="144" width="896" height="52" rx="12" fill="#ffffff" stroke="#dce1ed" stroke-width="1.3" data-node-box="true"/>
<circle cx="62" cy="161" r="6" fill="none" stroke="#58657c" stroke-width="1.8"/>
<path d="M51,184 L51,181 L55,174 L69,174 L73,181 L73,184" fill="none" stroke="#58657c" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="94" y="177" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Investigate the payments outage</text>
<rect x="720" y="156" width="192" height="28" rx="7" fill="#ede9fe" stroke="none" stroke-width="1.3"/>
<text x="816.0" y="175" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#5b21b6" text-anchor="middle">A2A via gateway</text>
</g>
<rect x="674" y="366" width="270" height="180" rx="16" fill="#edf4ff" stroke="#9dbdf6" stroke-width="1.3"/>
<rect x="685" y="350" width="247" height="30" rx="5" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<circle cx="700" cy="365" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="700" y="370" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">03</text>
<text x="722" y="371" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#1d4ed8" text-anchor="start" letter-spacing="0.3">WORKLOAD IDENTITY</text>
<path d="M284,302 L340,302" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="312" y="290" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="middle">calls</text>
<path d="M612,286 L680,286" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="272" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">LLM</text>
<path d="M612,452 L680,452" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="438" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">MCP</text>
<path d="M156,380 L156,418" fill="none" stroke="#776b9f" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="176" y="408" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="start">outside check</text>
<path d="M284,354 L314,354 L314,580 L638,580" fill="none" stroke="#be3455" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 6" data-connector="true"/>
<text x="362" y="567" font-family="Arial, sans-serif" font-size="18" font-weight="700" fill="#be3455" text-anchor="start">Direct MCP call</text>
<circle cx="653" cy="580" r="12" fill="#fff1f2" stroke="#be3455" stroke-width="2"/>
<path d="M648,575 L658,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M658,575 L648,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<text x="679" y="586" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#be3455" text-anchor="start" letter-spacing="0.5">REFUSED BY ISTIO</text>
<g data-node="agent">
<rect x="32" y="226" width="250" height="152" rx="12" fill="#ffffff" stroke="#dcd5ee" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="50" y="246" width="42" height="42" rx="10" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<rect x="57" y="257" width="28" height="22" rx="6" fill="none" stroke="#7c3aed" stroke-width="1.3"/>
<path d="M71,250 L71,257" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="71" cy="250" r="2.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="65" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="77" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<path d="M65,273 L77,273" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="105" y="275" font-family="Arial, sans-serif" font-size="29" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="52" y="314" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#172033" text-anchor="start">Incident-response agent</text>
<rect x="52" y="332" width="171" height="28" rx="7" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<text x="137.5" y="351" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#6d28d9" text-anchor="middle">Declarative Agent</text>
</g>
<g data-node="gateway">
<rect x="352" y="226" width="258" height="316" rx="12" fill="url(#defence-gateway-fill)" stroke="#5b4691" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M388,246 L404,252 L404,264 L400,273 L388,283 L376,273 L372,264 L372,252 L388,246" fill="none" stroke="#c4b5fd" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M382,263 L387,268 L395,257" fill="none" stroke="#ddd6fe" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/>
<text x="418" y="270" font-family="Arial, sans-serif" font-size="25" font-weight="700" fill="#ffffff" text-anchor="start">agentgateway</text>
<text x="374" y="294" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#c4b5fd" text-anchor="start">Policy enforcement</text>
<rect x="370" y="310" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="328" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="333" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">04</text>
<text x="414" y="335" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Caller identity</text>
<rect x="370" y="353" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="371" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="376" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">05</text>
<text x="414" y="378" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Tool permissions</text>
<rect x="370" y="396" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="414" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="419" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">06</text>
<text x="414" y="421" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Personal data</text>
<rect x="370" y="439" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="457" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="462" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">07</text>
<text x="414" y="464" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Request rate</text>
<path d="M374,491 L589,491" fill="none" stroke="#5a4b7c" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="391" cy="514" r="13" fill="#3e355f" stroke="none" stroke-width="1.5"/>
<text x="391" y="519" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#d5cce9" text-anchor="middle">08</text>
<text x="414" y="521" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#ded7ef" text-anchor="start">Logs + metrics</text>
</g>
<g data-node="claude">
<rect x="692" y="226" width="236" height="120" rx="12" fill="#fffaf5" stroke="#ead9cb" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M729.0,261.0 L740.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,263.5 L737.856406460551,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,265.3301270189222 L732.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,266.0 L724.0,277.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,265.3301270189222 L716.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,263.5 L710.143593539449,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.0,261.0 L708.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,258.5 L710.143593539449,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,256.6698729810778 L716.0,247.143593539449" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,256.0 L724.0,245.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,256.6698729810778 L732.0,247.14359353944897" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,258.5 L737.856406460551,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<text x="754" y="270" font-family="Arial, sans-serif" font-size="27" font-weight="700" fill="#643923" text-anchor="start">Claude</text>
<text x="712" y="310" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Haiku 4.5</text>
<text x="907" y="333" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#946e56" text-anchor="end">Anthropic</text>
</g>
<g data-node="tools">
<rect x="692" y="386" width="236" height="142" rx="12" fill="#ffffff" stroke="#c9d7ed" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="712" y="405" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="409" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="416" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="420" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="427" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="431" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<text x="754" y="429" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="712" y="467" font-family="Arial, sans-serif" font-size="23" font-weight="400" fill="#172033" text-anchor="start">Incident tools</text>
<text x="712" y="503" font-family="Menlo, Consolas, monospace" font-size="17" font-weight="400" fill="#58657c" text-anchor="start">MCPServer</text>
</g>
<g data-node="egress">
<rect x="32" y="430" width="250" height="112" rx="12" fill="#eff6ff" stroke="#c4d7f5" stroke-width="1.3" data-node-box="true"/>
<path d="M53,469 L65,445 L65,469 L53,469" fill="#60a5fa" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M69,450 L78,469 L69,469 L69,450" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M50,475 L80,475 L75,480 L55,480 L50,475" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="96" y="471" font-family="Arial, sans-serif" font-size="23" font-weight="700" fill="#1e40af" text-anchor="start">Istio egress</text>
<circle cx="253" cy="449" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="253" y="454" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">02</text>
<text x="52" y="515" font-family="Arial, sans-serif" font-size="19" font-weight="400" fill="#365a8c" text-anchor="start">example.com is refused</text>
</g>
<text x="32" y="631" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#172033" text-anchor="start">Ask the agent to close every incident</text>
<text x="32" y="658" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#58657c" text-anchor="start">Closing the record does not resolve the fault.</text>
<g data-node="before">
<rect x="32" y="680" width="436" height="166" rx="12" fill="#fff5f6" stroke="#f0bdc9" stroke-width="1.3" data-node-box="true"/>
<text x="52" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#a82e4b" text-anchor="start" letter-spacing="0.8">WITHOUT A TOOL POLICY</text>
<text x="52" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#172033" text-anchor="start">0</text>
<text x="85" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#58657c" text-anchor="start">open</text>
<path d="M216,727 L216,768" fill="none" stroke="#eccbd3" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="244" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#be3455" text-anchor="start">3</text>
<text x="279" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#be3455" text-anchor="start">closed</text>
<text x="52" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#9f2946" text-anchor="start">3 faults still unresolved</text>
<text x="52" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#9f6270" text-anchor="start">Bulk close succeeded</text>
</g>
<g data-node="after">
<rect x="492" y="680" width="436" height="166" rx="12" fill="#effbf5" stroke="#a6dac1" stroke-width="1.3" data-node-box="true"/>
<text x="512" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#087f5b" text-anchor="start" letter-spacing="0.8">WITH A READ-ONLY POLICY</text>
<text x="512" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#087f5b" text-anchor="start">3</text>
<text x="547" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#087f5b" text-anchor="start">open</text>
<rect x="741" y="731" width="166" height="28" rx="7" fill="#d0f0df" stroke="none" stroke-width="1.3"/>
<text x="824.0" y="750" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#086b4e" text-anchor="middle">WRITE REFUSED</text>
<text x="512" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#086b4e" text-anchor="start">Read-only tools still work</text>
<text x="512" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#46735f" text-anchor="start">Bulk close refused</text>
</g>
<text x="32" y="881" font-family="Arial, sans-serif" font-size="16" font-weight="400" fill="#58657c" text-anchor="start">Illustrated chapter-5 outcome. Live incident records and evidence are in the App tab.</text>
</svg>

### Create the namespaces

Enrol the lab's agent, tool and gateway namespaces in ambient.

**Expected:** Three dd- namespaces are created.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: v1
kind: Namespace
metadata:
  name: dd-agents
  labels: {istio.io/dataplane-mode: ambient}
---
apiVersion: v1
kind: Namespace
metadata:
  name: dd-tools
  labels: {istio.io/dataplane-mode: ambient}
---
apiVersion: v1
kind: Namespace
metadata:
  name: dd-gateway
  labels: {istio.io/dataplane-mode: ambient}
YAML

### Give the agent an identity

Create the lab signing key and store Alice's gateway token in a Secret for kagent to reference. Copy the installed Anthropic credential into the lab gateway's namespace without printing it.

**Expected:** The agent identity and gateway provider credential are stored separately.

In [ ]:
python3 "$DD/identity.py" setup
ALICE=$(python3 "$DD/identity.py" token --user alice)
kubectl --context kind-mesh1 -n dd-agents create secret generic dd-agent-identity --from-literal="token=$ALICE" --from-literal="authorization=Bearer $ALICE" --dry-run=client -o yaml | kubectl --context kind-mesh1 apply -f -
kubectl --context kind-mesh1 -n agentgateway-system get secret anthropic-secret -o json | jq '{apiVersion:"v1",kind:"Secret",metadata:{name:"dd-anthropic",namespace:"dd-gateway"},type:"Opaque",data:.data}' | kubectl --context kind-mesh1 apply -f -

### Run the incident tool server

Build the server for reading and changing the lab's incident records, then let kagent deploy it.

**Expected:** The image's record-mutation tests pass and defence-operations reports Ready.

In [ ]:
docker build -t localhost:5001/defence-ops-mcp:v1 "$DD/operations-mcp"
docker push localhost:5001/defence-ops-mcp:v1
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: kagent.dev/v1alpha1
kind: MCPServer
metadata: {name: defence-operations, namespace: dd-tools}
spec:
  deployment:
    image: localhost:5001/defence-ops-mcp:v1
    imagePullPolicy: Always
    port: 3000
    resources:
      requests: {cpu: 10m, memory: 32Mi}
      limits: {cpu: 200m, memory: 128Mi}
  transportType: http
  httpTransport: {path: /mcp, targetPort: 3000}
YAML
kubectl --context kind-mesh1 -n dd-tools wait mcpserver/defence-operations --for=condition=Ready --timeout=120s

### Route model, tool and agent traffic

Create the lab gateway. The model backend calls Anthropic, the MCP backend reaches the kagent-managed server, and the A2A route exposes the agent's runtime.

**Expected:** dd-gateway rolls out. The agent route starts serving after the next step creates the Agent.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: {name: dd-gateway, namespace: dd-gateway}
spec:
  gatewayClassName: enterprise-agentgateway
  listeners:
  - name: http
    protocol: HTTP
    port: 80
    allowedRoutes: {namespaces: {from: Same}}
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: {name: model, namespace: dd-gateway}
spec:
  ai:
    provider:
      anthropic: {model: claude-haiku-4-5}
  policies:
    auth:
      secretRef: {name: dd-anthropic}
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: {name: tools, namespace: dd-gateway}
spec:
  mcp:
    targets:
    - name: operations
      static: {host: defence-operations.dd-tools.svc.cluster.local, port: 3000, path: /mcp}
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: model, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: PathPrefix, value: /v1/chat/completions}}]
    backendRefs:
    - {name: model, group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend}
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: tools, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: PathPrefix, value: /mcp}}]
    backendRefs:
    - {name: tools, group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend}
---
# The lab client reaches the real kagent agent through its A2A endpoint.
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: agent, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: PathPrefix, value: /a2a}}]
    filters:
    - type: URLRewrite
      urlRewrite:
        path: {type: ReplacePrefixMatch, replacePrefixMatch: /}
    backendRefs:
    - {name: defence-agent, namespace: dd-agents, port: 8080}
---
apiVersion: gateway.networking.k8s.io/v1beta1
kind: ReferenceGrant
metadata: {name: dd-gateway-to-agent, namespace: dd-agents}
spec:
  from:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, namespace: dd-gateway}
  to:
  - {group: "", kind: Service, name: defence-agent}
---
# Read-only incident state for the console. JWT authentication on the gateway
# applies here too. Mutations are MCP tools, not public HTTP write endpoints.
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: incident-state, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: Exact, value: /ops-state}}]
    filters:
    - type: URLRewrite
      urlRewrite:
        path: {type: ReplaceFullPath, replaceFullPath: /state}
    backendRefs:
    - {name: defence-operations, namespace: dd-tools, port: 3000}
---
apiVersion: gateway.networking.k8s.io/v1beta1
kind: ReferenceGrant
metadata: {name: dd-gateway-to-state, namespace: dd-tools}
spec:
  from:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, namespace: dd-gateway}
  to:
  - {group: "", kind: Service, name: defence-operations}
YAML
kubectl --context kind-mesh1 -n dd-gateway rollout status deploy/dd-gateway --timeout=120s

### Create the kagent Agent

The ModelConfig uses agentgateway's OpenAI-compatible endpoint. The RemoteMCPServer registers the gateway's tools. The Agent references both.

**Expected:** kagent accepts the Agent and its runtime becomes Ready.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: kagent.dev/v1alpha2
kind: ModelConfig
metadata: {name: defence-model, namespace: dd-agents}
spec:
  provider: OpenAI
  model: claude-haiku-4-5
  apiKeySecret: dd-agent-identity
  apiKeySecretKey: token
  openAI:
    baseUrl: http://dd-gateway.dd-gateway.svc/v1
    maxTokens: 256
    temperature: "0"
---
apiVersion: kagent.dev/v1alpha2
kind: RemoteMCPServer
metadata: {name: defence-tools, namespace: dd-agents}
spec:
  description: Incident operations reached through agentgateway
  protocol: STREAMABLE_HTTP
  url: http://dd-gateway.dd-gateway.svc/mcp
  headersFrom:
  - name: Authorization
    valueFrom: {type: Secret, name: dd-agent-identity, key: authorization}
---
apiVersion: kagent.dev/v1alpha2
kind: Agent
metadata: {name: defence-agent, namespace: dd-agents}
spec:
  type: Declarative
  description: An incident-response agent whose operational permissions are enforced by agentgateway
  declarative:
    modelConfig: defence-model
    runtime: python
    systemMessage: |
      You are an incident-response assistant. Use tools to investigate incidents
      and carry out requested operations on the incident records.
      These are disposable lab records. Execute explicitly requested operations
      when the tool is available, without asking for another confirmation.
      For incident questions, always read the records with a tool. For an action,
      call the named tool and report what it actually changed. Do not claim that
      closing an incident resolves its underlying fault. Do not invent records.
      If a requested tool is unavailable or a call is refused, say so and stop.
      Do not substitute another tool or claim the operation succeeded.
      For a request to repeat text or reply with a word, reply directly.
      Keep replies to one paragraph.
    tools:
    - type: McpServer
      mcpServer:
        kind: RemoteMCPServer
        apiGroup: kagent.dev
        name: defence-tools
        toolNames: [list_incidents, get_incident, close_all_incidents, delete_incident_history]
    deployment:
      resources:
        requests: {cpu: 25m, memory: 128Mi}
        limits: {cpu: 500m, memory: 512Mi}
YAML
kubectl --context kind-mesh1 -n dd-agents wait agent/defence-agent --for=condition=Ready --timeout=180s

### Ask the agent to investigate

Send an A2A message asking for the current incidents. Read the actual tool call and records returned by the agent, then check that kagent owns its Deployment.

**Expected:** The agent reads three open incidents affecting payments, customer-api and settlement. Its Deployment is owned by Agent/defence-agent.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
kubectl --context kind-mesh1 -n dd-agents get deployment defence-agent -o json | jq '{owner:.metadata.ownerReferences[0].kind,name:.metadata.ownerReferences[0].name,serviceAccount:.spec.template.spec.serviceAccountName}'

### Declare the outside destination

Put example.com behind an Istio waypoint visible only to dd-agents. This is a direct egress check from the kagent runtime, not an agent turn.

**Expected:** The waypoint is Ready and HTTPS to example.com returns 200.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
# Only this namespace sees the ServiceEntry. Other labs keep their DNS answers.
# This covers example.com, not arbitrary internet destinations or IP literals.
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: dd-egress
  namespace: dd-agents
  labels: {istio.io/waypoint-for: service}
spec:
  gatewayClassName: istio-waypoint
  listeners: [{name: mesh, port: 15008, protocol: HBONE}]
---
apiVersion: networking.istio.io/v1
kind: ServiceEntry
metadata:
  name: outside-example
  namespace: dd-agents
  labels: {istio.io/use-waypoint: dd-egress}
spec:
  exportTo: ["."]
  hosts: [example.com]
  ports: [{number: 443, name: tls, protocol: TLS}]
  resolution: DNS
  location: MESH_EXTERNAL
YAML
kubectl --context kind-mesh1 -n dd-agents rollout status deploy/dd-egress --timeout=120s
kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 --retry 10 --retry-all-errors --retry-delay 1 -o /dev/null -w 'Outside HTTP %{http_code}\n' https://example.com

### Select the installed rate limiter

The default Service on this rig also selects the waypoint limiter, which has a separate counter. Create a lab-owned Service selecting the main limiter's active ReplicaSet.

**Expected:** dd-rate-limiter selects the main limiter without changing its Deployment.

In [ ]:
HASH=$(kubectl --context kind-mesh1 -n agentgateway-system get rs -o json | jq -er '[.items[] | select(any(.metadata.ownerReferences[]?; .kind == "Deployment" and .name == "rate-limiter-enterprise-agentgateway")) | select(.spec.replicas > 0)] | sort_by(.metadata.creationTimestamp) | last | .metadata.labels["pod-template-hash"]')
jq -n --arg hash "$HASH" '{apiVersion:"v1",kind:"Service",metadata:{name:"dd-rate-limiter",namespace:"agentgateway-system"},spec:{selector:{app:"rate-limiter","pod-template-hash":$hash},ports:[{name:"grpc",port:8083,targetPort:8083}]}}' | kubectl --context kind-mesh1 apply -f -
kubectl --context kind-mesh1 -n agentgateway-system get service dd-rate-limiter -o yaml

## 2. Control the agent's outside access

Check whether the kagent runtime can reach example.com. Add a deny policy to its egress waypoint, then repeat the request.

This checks a workload boundary directly. The request is not generated by the model. Kindnet does not enforce NetworkPolicy, so this chapter proves control of example.com only.

<svg id="defence-architecture" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 900" width="100%" role="img" aria-labelledby="defence-title defence-desc">
<title id="defence-title">Layers of defence for a kagent incident-response agent</title>
<desc id="defence-desc">kagent runs the agent and incident MCP tools. Agentgateway enforces caller identity, tool permissions, prompt guards and rate limits. Istio refuses direct tool access and example.com egress. Before the tool policy, three unresolved incidents are closed. After the policy, the bulk close is refused and investigation remains available.</desc>
<defs>
      <linearGradient id="defence-gateway-fill" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#382763"/><stop offset="1" stop-color="#231c40"/></linearGradient>
      <filter id="defence-shadow" x="-15%" y="-15%" width="130%" height="140%"><feDropShadow dx="0" dy="5" stdDeviation="6" flood-color="#232142" flood-opacity=".09"/></filter>
      <marker id="defence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M1 1 L9 5 L1 9 Z" fill="#776b9f"/></marker>
    </defs>
<rect x="0" y="0" width="960" height="900" rx="20" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<rect x="0" y="0" width="960" height="120" rx="20" fill="#101426" stroke="none" stroke-width="1.3"/>
<rect x="0" y="90" width="960" height="30" rx="0" fill="#101426" stroke="none" stroke-width="1.3"/>
<text x="32" y="34" font-family="Arial, sans-serif" font-size="13" font-weight="700" fill="#c4b5fd" text-anchor="start" letter-spacing="2">SOLO / INCIDENT RESPONSE</text>
<text x="32" y="74" font-family="Arial, sans-serif" font-size="36" font-weight="700" fill="#ffffff" text-anchor="start">Layers of defence</text>
<text x="32" y="103" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#cbd5e1" text-anchor="start">Where the agent&#x27;s permissions are enforced</text>
<path d="M808,30 L794,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M808,30 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M822,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="808" cy="30" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="794" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="822" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="808" cy="58" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<text x="838" y="51" font-family="Arial, sans-serif" font-size="19" font-weight="700" fill="#ffffff" text-anchor="start" letter-spacing="1">SOLO.IO</text>
<rect x="842" y="76" width="86" height="28" rx="7" fill="#272c46" stroke="none" stroke-width="1.3"/>
<text x="885.0" y="95" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#d8ddec" text-anchor="middle">mesh1</text>
<g data-node="request">
<rect x="32" y="144" width="896" height="52" rx="12" fill="#ffffff" stroke="#dce1ed" stroke-width="1.3" data-node-box="true"/>
<circle cx="62" cy="161" r="6" fill="none" stroke="#58657c" stroke-width="1.8"/>
<path d="M51,184 L51,181 L55,174 L69,174 L73,181 L73,184" fill="none" stroke="#58657c" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="94" y="177" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Investigate the payments outage</text>
<rect x="720" y="156" width="192" height="28" rx="7" fill="#ede9fe" stroke="none" stroke-width="1.3"/>
<text x="816.0" y="175" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#5b21b6" text-anchor="middle">A2A via gateway</text>
</g>
<rect x="674" y="366" width="270" height="180" rx="16" fill="#edf4ff" stroke="#9dbdf6" stroke-width="1.3"/>
<rect x="685" y="350" width="247" height="30" rx="5" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<circle cx="700" cy="365" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="700" y="370" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">03</text>
<text x="722" y="371" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#1d4ed8" text-anchor="start" letter-spacing="0.3">WORKLOAD IDENTITY</text>
<path d="M284,302 L340,302" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="312" y="290" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="middle">calls</text>
<path d="M612,286 L680,286" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="272" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">LLM</text>
<path d="M612,452 L680,452" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="438" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">MCP</text>
<path d="M156,380 L156,418" fill="none" stroke="#776b9f" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="176" y="408" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="start">outside check</text>
<path d="M284,354 L314,354 L314,580 L638,580" fill="none" stroke="#be3455" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 6" data-connector="true"/>
<text x="362" y="567" font-family="Arial, sans-serif" font-size="18" font-weight="700" fill="#be3455" text-anchor="start">Direct MCP call</text>
<circle cx="653" cy="580" r="12" fill="#fff1f2" stroke="#be3455" stroke-width="2"/>
<path d="M648,575 L658,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M658,575 L648,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<text x="679" y="586" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#be3455" text-anchor="start" letter-spacing="0.5">REFUSED BY ISTIO</text>
<g data-node="agent">
<rect x="32" y="226" width="250" height="152" rx="12" fill="#ffffff" stroke="#dcd5ee" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="50" y="246" width="42" height="42" rx="10" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<rect x="57" y="257" width="28" height="22" rx="6" fill="none" stroke="#7c3aed" stroke-width="1.3"/>
<path d="M71,250 L71,257" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="71" cy="250" r="2.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="65" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="77" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<path d="M65,273 L77,273" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="105" y="275" font-family="Arial, sans-serif" font-size="29" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="52" y="314" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#172033" text-anchor="start">Incident-response agent</text>
<rect x="52" y="332" width="171" height="28" rx="7" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<text x="137.5" y="351" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#6d28d9" text-anchor="middle">Declarative Agent</text>
</g>
<g data-node="gateway">
<rect x="352" y="226" width="258" height="316" rx="12" fill="url(#defence-gateway-fill)" stroke="#5b4691" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M388,246 L404,252 L404,264 L400,273 L388,283 L376,273 L372,264 L372,252 L388,246" fill="none" stroke="#c4b5fd" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M382,263 L387,268 L395,257" fill="none" stroke="#ddd6fe" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/>
<text x="418" y="270" font-family="Arial, sans-serif" font-size="25" font-weight="700" fill="#ffffff" text-anchor="start">agentgateway</text>
<text x="374" y="294" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#c4b5fd" text-anchor="start">Policy enforcement</text>
<rect x="370" y="310" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="328" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="333" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">04</text>
<text x="414" y="335" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Caller identity</text>
<rect x="370" y="353" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="371" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="376" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">05</text>
<text x="414" y="378" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Tool permissions</text>
<rect x="370" y="396" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="414" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="419" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">06</text>
<text x="414" y="421" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Personal data</text>
<rect x="370" y="439" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="457" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="462" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">07</text>
<text x="414" y="464" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Request rate</text>
<path d="M374,491 L589,491" fill="none" stroke="#5a4b7c" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="391" cy="514" r="13" fill="#3e355f" stroke="none" stroke-width="1.5"/>
<text x="391" y="519" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#d5cce9" text-anchor="middle">08</text>
<text x="414" y="521" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#ded7ef" text-anchor="start">Logs + metrics</text>
</g>
<g data-node="claude">
<rect x="692" y="226" width="236" height="120" rx="12" fill="#fffaf5" stroke="#ead9cb" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M729.0,261.0 L740.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,263.5 L737.856406460551,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,265.3301270189222 L732.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,266.0 L724.0,277.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,265.3301270189222 L716.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,263.5 L710.143593539449,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.0,261.0 L708.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,258.5 L710.143593539449,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,256.6698729810778 L716.0,247.143593539449" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,256.0 L724.0,245.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,256.6698729810778 L732.0,247.14359353944897" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,258.5 L737.856406460551,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<text x="754" y="270" font-family="Arial, sans-serif" font-size="27" font-weight="700" fill="#643923" text-anchor="start">Claude</text>
<text x="712" y="310" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Haiku 4.5</text>
<text x="907" y="333" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#946e56" text-anchor="end">Anthropic</text>
</g>
<g data-node="tools">
<rect x="692" y="386" width="236" height="142" rx="12" fill="#ffffff" stroke="#c9d7ed" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="712" y="405" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="409" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="416" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="420" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="427" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="431" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<text x="754" y="429" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="712" y="467" font-family="Arial, sans-serif" font-size="23" font-weight="400" fill="#172033" text-anchor="start">Incident tools</text>
<text x="712" y="503" font-family="Menlo, Consolas, monospace" font-size="17" font-weight="400" fill="#58657c" text-anchor="start">MCPServer</text>
</g>
<g data-node="egress">
<rect x="32" y="430" width="250" height="112" rx="12" fill="#eff6ff" stroke="#c4d7f5" stroke-width="1.3" data-node-box="true"/>
<path d="M53,469 L65,445 L65,469 L53,469" fill="#60a5fa" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M69,450 L78,469 L69,469 L69,450" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M50,475 L80,475 L75,480 L55,480 L50,475" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="96" y="471" font-family="Arial, sans-serif" font-size="23" font-weight="700" fill="#1e40af" text-anchor="start">Istio egress</text>
<circle cx="253" cy="449" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="253" y="454" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">02</text>
<text x="52" y="515" font-family="Arial, sans-serif" font-size="19" font-weight="400" fill="#365a8c" text-anchor="start">example.com is refused</text>
</g>
<text x="32" y="631" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#172033" text-anchor="start">Ask the agent to close every incident</text>
<text x="32" y="658" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#58657c" text-anchor="start">Closing the record does not resolve the fault.</text>
<g data-node="before">
<rect x="32" y="680" width="436" height="166" rx="12" fill="#fff5f6" stroke="#f0bdc9" stroke-width="1.3" data-node-box="true"/>
<text x="52" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#a82e4b" text-anchor="start" letter-spacing="0.8">WITHOUT A TOOL POLICY</text>
<text x="52" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#172033" text-anchor="start">0</text>
<text x="85" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#58657c" text-anchor="start">open</text>
<path d="M216,727 L216,768" fill="none" stroke="#eccbd3" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="244" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#be3455" text-anchor="start">3</text>
<text x="279" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#be3455" text-anchor="start">closed</text>
<text x="52" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#9f2946" text-anchor="start">3 faults still unresolved</text>
<text x="52" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#9f6270" text-anchor="start">Bulk close succeeded</text>
</g>
<g data-node="after">
<rect x="492" y="680" width="436" height="166" rx="12" fill="#effbf5" stroke="#a6dac1" stroke-width="1.3" data-node-box="true"/>
<text x="512" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#087f5b" text-anchor="start" letter-spacing="0.8">WITH A READ-ONLY POLICY</text>
<text x="512" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#087f5b" text-anchor="start">3</text>
<text x="547" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#087f5b" text-anchor="start">open</text>
<rect x="741" y="731" width="166" height="28" rx="7" fill="#d0f0df" stroke="none" stroke-width="1.3"/>
<text x="824.0" y="750" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#086b4e" text-anchor="middle">WRITE REFUSED</text>
<text x="512" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#086b4e" text-anchor="start">Read-only tools still work</text>
<text x="512" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#46735f" text-anchor="start">Bulk close refused</text>
</g>
<text x="32" y="881" font-family="Arial, sans-serif" font-size="16" font-weight="400" fill="#58657c" text-anchor="start">Illustrated chapter-5 outcome. Live incident records and evidence are in the App tab.</text>
</svg>

### Without the control

Fetch example.com from the kagent agent's managed runtime.

**Expected:** The HTTPS request returns 200.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 -o /dev/null -w 'Outside HTTP %{http_code}\n' https://example.com

### Add the control

Deny traffic handled by this lab's egress waypoint.

**Expected:** The AuthorizationPolicy is applied.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: {name: outside-deny, namespace: dd-agents}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: Gateway, name: dd-egress}
  action: DENY
  rules: [{}]
YAML
sleep 2

### With the control

Repeat the direct egress check and read the waypoint's RBAC counter. Then ask the agent to investigate incidents through its permitted model and tool paths.

**Expected:** Outside access is refused and recorded. The agent still completes its tool call.

In [ ]:
if kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 -o /dev/null https://example.com; then printf 'Unexpected outside access\n'; exit 1; else printf 'Outside request refused\n'; fi
kubectl --context kind-mesh1 -n dd-agents exec deploy/dd-egress -- pilot-agent request GET 'stats?filter=rbac'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

## 3. Stop the agent bypassing the tool gateway

The incident tools are managed by kagent. Without a workload policy, the agent runtime can also call them directly. Require agent traffic to pass through agentgateway.

Istio checks SPIFFE identity. The gateway is admitted, and kagent's controller keeps direct access for tool discovery. The agent's own identity is refused on the direct path.

<svg id="defence-architecture" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 900" width="100%" role="img" aria-labelledby="defence-title defence-desc">
<title id="defence-title">Layers of defence for a kagent incident-response agent</title>
<desc id="defence-desc">kagent runs the agent and incident MCP tools. Agentgateway enforces caller identity, tool permissions, prompt guards and rate limits. Istio refuses direct tool access and example.com egress. Before the tool policy, three unresolved incidents are closed. After the policy, the bulk close is refused and investigation remains available.</desc>
<defs>
      <linearGradient id="defence-gateway-fill" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#382763"/><stop offset="1" stop-color="#231c40"/></linearGradient>
      <filter id="defence-shadow" x="-15%" y="-15%" width="130%" height="140%"><feDropShadow dx="0" dy="5" stdDeviation="6" flood-color="#232142" flood-opacity=".09"/></filter>
      <marker id="defence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M1 1 L9 5 L1 9 Z" fill="#776b9f"/></marker>
    </defs>
<rect x="0" y="0" width="960" height="900" rx="20" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<rect x="0" y="0" width="960" height="120" rx="20" fill="#101426" stroke="none" stroke-width="1.3"/>
<rect x="0" y="90" width="960" height="30" rx="0" fill="#101426" stroke="none" stroke-width="1.3"/>
<text x="32" y="34" font-family="Arial, sans-serif" font-size="13" font-weight="700" fill="#c4b5fd" text-anchor="start" letter-spacing="2">SOLO / INCIDENT RESPONSE</text>
<text x="32" y="74" font-family="Arial, sans-serif" font-size="36" font-weight="700" fill="#ffffff" text-anchor="start">Layers of defence</text>
<text x="32" y="103" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#cbd5e1" text-anchor="start">Where the agent&#x27;s permissions are enforced</text>
<path d="M808,30 L794,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M808,30 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M822,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="808" cy="30" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="794" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="822" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="808" cy="58" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<text x="838" y="51" font-family="Arial, sans-serif" font-size="19" font-weight="700" fill="#ffffff" text-anchor="start" letter-spacing="1">SOLO.IO</text>
<rect x="842" y="76" width="86" height="28" rx="7" fill="#272c46" stroke="none" stroke-width="1.3"/>
<text x="885.0" y="95" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#d8ddec" text-anchor="middle">mesh1</text>
<g data-node="request">
<rect x="32" y="144" width="896" height="52" rx="12" fill="#ffffff" stroke="#dce1ed" stroke-width="1.3" data-node-box="true"/>
<circle cx="62" cy="161" r="6" fill="none" stroke="#58657c" stroke-width="1.8"/>
<path d="M51,184 L51,181 L55,174 L69,174 L73,181 L73,184" fill="none" stroke="#58657c" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="94" y="177" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Investigate the payments outage</text>
<rect x="720" y="156" width="192" height="28" rx="7" fill="#ede9fe" stroke="none" stroke-width="1.3"/>
<text x="816.0" y="175" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#5b21b6" text-anchor="middle">A2A via gateway</text>
</g>
<rect x="674" y="366" width="270" height="180" rx="16" fill="#edf4ff" stroke="#9dbdf6" stroke-width="1.3"/>
<rect x="685" y="350" width="247" height="30" rx="5" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<circle cx="700" cy="365" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="700" y="370" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">03</text>
<text x="722" y="371" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#1d4ed8" text-anchor="start" letter-spacing="0.3">WORKLOAD IDENTITY</text>
<path d="M284,302 L340,302" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="312" y="290" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="middle">calls</text>
<path d="M612,286 L680,286" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="272" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">LLM</text>
<path d="M612,452 L680,452" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="438" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">MCP</text>
<path d="M156,380 L156,418" fill="none" stroke="#776b9f" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="176" y="408" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="start">outside check</text>
<path d="M284,354 L314,354 L314,580 L638,580" fill="none" stroke="#be3455" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 6" data-connector="true"/>
<text x="362" y="567" font-family="Arial, sans-serif" font-size="18" font-weight="700" fill="#be3455" text-anchor="start">Direct MCP call</text>
<circle cx="653" cy="580" r="12" fill="#fff1f2" stroke="#be3455" stroke-width="2"/>
<path d="M648,575 L658,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M658,575 L648,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<text x="679" y="586" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#be3455" text-anchor="start" letter-spacing="0.5">REFUSED BY ISTIO</text>
<g data-node="agent">
<rect x="32" y="226" width="250" height="152" rx="12" fill="#ffffff" stroke="#dcd5ee" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="50" y="246" width="42" height="42" rx="10" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<rect x="57" y="257" width="28" height="22" rx="6" fill="none" stroke="#7c3aed" stroke-width="1.3"/>
<path d="M71,250 L71,257" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="71" cy="250" r="2.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="65" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="77" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<path d="M65,273 L77,273" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="105" y="275" font-family="Arial, sans-serif" font-size="29" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="52" y="314" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#172033" text-anchor="start">Incident-response agent</text>
<rect x="52" y="332" width="171" height="28" rx="7" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<text x="137.5" y="351" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#6d28d9" text-anchor="middle">Declarative Agent</text>
</g>
<g data-node="gateway">
<rect x="352" y="226" width="258" height="316" rx="12" fill="url(#defence-gateway-fill)" stroke="#5b4691" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M388,246 L404,252 L404,264 L400,273 L388,283 L376,273 L372,264 L372,252 L388,246" fill="none" stroke="#c4b5fd" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M382,263 L387,268 L395,257" fill="none" stroke="#ddd6fe" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/>
<text x="418" y="270" font-family="Arial, sans-serif" font-size="25" font-weight="700" fill="#ffffff" text-anchor="start">agentgateway</text>
<text x="374" y="294" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#c4b5fd" text-anchor="start">Policy enforcement</text>
<rect x="370" y="310" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="328" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="333" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">04</text>
<text x="414" y="335" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Caller identity</text>
<rect x="370" y="353" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="371" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="376" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">05</text>
<text x="414" y="378" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Tool permissions</text>
<rect x="370" y="396" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="414" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="419" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">06</text>
<text x="414" y="421" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Personal data</text>
<rect x="370" y="439" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="457" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="462" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">07</text>
<text x="414" y="464" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Request rate</text>
<path d="M374,491 L589,491" fill="none" stroke="#5a4b7c" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="391" cy="514" r="13" fill="#3e355f" stroke="none" stroke-width="1.5"/>
<text x="391" y="519" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#d5cce9" text-anchor="middle">08</text>
<text x="414" y="521" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#ded7ef" text-anchor="start">Logs + metrics</text>
</g>
<g data-node="claude">
<rect x="692" y="226" width="236" height="120" rx="12" fill="#fffaf5" stroke="#ead9cb" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M729.0,261.0 L740.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,263.5 L737.856406460551,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,265.3301270189222 L732.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,266.0 L724.0,277.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,265.3301270189222 L716.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,263.5 L710.143593539449,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.0,261.0 L708.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,258.5 L710.143593539449,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,256.6698729810778 L716.0,247.143593539449" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,256.0 L724.0,245.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,256.6698729810778 L732.0,247.14359353944897" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,258.5 L737.856406460551,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<text x="754" y="270" font-family="Arial, sans-serif" font-size="27" font-weight="700" fill="#643923" text-anchor="start">Claude</text>
<text x="712" y="310" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Haiku 4.5</text>
<text x="907" y="333" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#946e56" text-anchor="end">Anthropic</text>
</g>
<g data-node="tools">
<rect x="692" y="386" width="236" height="142" rx="12" fill="#ffffff" stroke="#c9d7ed" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="712" y="405" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="409" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="416" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="420" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="427" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="431" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<text x="754" y="429" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="712" y="467" font-family="Arial, sans-serif" font-size="23" font-weight="400" fill="#172033" text-anchor="start">Incident tools</text>
<text x="712" y="503" font-family="Menlo, Consolas, monospace" font-size="17" font-weight="400" fill="#58657c" text-anchor="start">MCPServer</text>
</g>
<g data-node="egress">
<rect x="32" y="430" width="250" height="112" rx="12" fill="#eff6ff" stroke="#c4d7f5" stroke-width="1.3" data-node-box="true"/>
<path d="M53,469 L65,445 L65,469 L53,469" fill="#60a5fa" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M69,450 L78,469 L69,469 L69,450" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M50,475 L80,475 L75,480 L55,480 L50,475" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="96" y="471" font-family="Arial, sans-serif" font-size="23" font-weight="700" fill="#1e40af" text-anchor="start">Istio egress</text>
<circle cx="253" cy="449" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="253" y="454" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">02</text>
<text x="52" y="515" font-family="Arial, sans-serif" font-size="19" font-weight="400" fill="#365a8c" text-anchor="start">example.com is refused</text>
</g>
<text x="32" y="631" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#172033" text-anchor="start">Ask the agent to close every incident</text>
<text x="32" y="658" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#58657c" text-anchor="start">Closing the record does not resolve the fault.</text>
<g data-node="before">
<rect x="32" y="680" width="436" height="166" rx="12" fill="#fff5f6" stroke="#f0bdc9" stroke-width="1.3" data-node-box="true"/>
<text x="52" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#a82e4b" text-anchor="start" letter-spacing="0.8">WITHOUT A TOOL POLICY</text>
<text x="52" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#172033" text-anchor="start">0</text>
<text x="85" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#58657c" text-anchor="start">open</text>
<path d="M216,727 L216,768" fill="none" stroke="#eccbd3" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="244" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#be3455" text-anchor="start">3</text>
<text x="279" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#be3455" text-anchor="start">closed</text>
<text x="52" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#9f2946" text-anchor="start">3 faults still unresolved</text>
<text x="52" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#9f6270" text-anchor="start">Bulk close succeeded</text>
</g>
<g data-node="after">
<rect x="492" y="680" width="436" height="166" rx="12" fill="#effbf5" stroke="#a6dac1" stroke-width="1.3" data-node-box="true"/>
<text x="512" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#087f5b" text-anchor="start" letter-spacing="0.8">WITH A READ-ONLY POLICY</text>
<text x="512" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#087f5b" text-anchor="start">3</text>
<text x="547" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#087f5b" text-anchor="start">open</text>
<rect x="741" y="731" width="166" height="28" rx="7" fill="#d0f0df" stroke="none" stroke-width="1.3"/>
<text x="824.0" y="750" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#086b4e" text-anchor="middle">WRITE REFUSED</text>
<text x="512" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#086b4e" text-anchor="start">Read-only tools still work</text>
<text x="512" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#46735f" text-anchor="start">Bulk close refused</text>
</g>
<text x="32" y="881" font-family="Arial, sans-serif" font-size="16" font-weight="400" fill="#58657c" text-anchor="start">Illustrated chapter-5 outcome. Live incident records and evidence are in the App tab.</text>
</svg>

### Without the control

Make a direct MCP call from the kagent runtime to the operations Service, bypassing agentgateway.

**Expected:** The incident records are returned directly.

In [ ]:
dd_mcp "$ALICE" tools/call '{"name":"list_incidents","arguments":{}}' http://defence-operations.dd-tools.svc:3000/mcp

### Add the control

Allow the gateway and kagent's discovery controller into the tool namespace. Do not admit the agent's ServiceAccount.

**Expected:** gateway-only is applied in dd-tools.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
# Agent traffic must pass through agentgateway. kagent's controller also needs
# direct access to discover the tools of the MCPServer it manages.
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: {name: gateway-only, namespace: dd-tools}
spec:
  action: ALLOW
  rules:
  - from:
    - source:
        principals:
        - mesh1/ns/dd-gateway/sa/dd-gateway
        - mesh1/ns/kagent/sa/kagent-controller
YAML
sleep 2

### With the control

Repeat the direct MCP call and read the ztunnel refusal. Then ask the agent to investigate through its configured gateway path.

**Expected:** Direct access is refused under the defence-agent SPIFFE identity. The agent's normal tool call still reads the incidents.

In [ ]:
if dd_mcp "$ALICE" tools/call '{"name":"list_incidents","arguments":{}}' http://defence-operations.dd-tools.svc:3000/mcp; then printf 'Unexpected direct tool access\n'; exit 1; else printf 'Direct tool request refused\n'; fi
kubectl --context kind-mesh1 -n istio-system logs -l app=ztunnel --since=30s --tail=3000 | jq -c 'select(.["src.namespace"] == "dd-agents" and .direction == "inbound" and (.error // "" | contains("policy rejection"))) | {source:.["src.identity"],destination:.["dst.namespace"],error}'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

## 4. Require a signed caller token

The gateway currently accepts a model request without a token. Add Strict JWT authentication, then compare the anonymous request with the agent's authenticated turn.

The agent's ModelConfig and RemoteMCPServer reference its identity Secret. The Anthropic provider key is held only by agentgateway.

<svg id="defence-architecture" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 900" width="100%" role="img" aria-labelledby="defence-title defence-desc">
<title id="defence-title">Layers of defence for a kagent incident-response agent</title>
<desc id="defence-desc">kagent runs the agent and incident MCP tools. Agentgateway enforces caller identity, tool permissions, prompt guards and rate limits. Istio refuses direct tool access and example.com egress. Before the tool policy, three unresolved incidents are closed. After the policy, the bulk close is refused and investigation remains available.</desc>
<defs>
      <linearGradient id="defence-gateway-fill" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#382763"/><stop offset="1" stop-color="#231c40"/></linearGradient>
      <filter id="defence-shadow" x="-15%" y="-15%" width="130%" height="140%"><feDropShadow dx="0" dy="5" stdDeviation="6" flood-color="#232142" flood-opacity=".09"/></filter>
      <marker id="defence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M1 1 L9 5 L1 9 Z" fill="#776b9f"/></marker>
    </defs>
<rect x="0" y="0" width="960" height="900" rx="20" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<rect x="0" y="0" width="960" height="120" rx="20" fill="#101426" stroke="none" stroke-width="1.3"/>
<rect x="0" y="90" width="960" height="30" rx="0" fill="#101426" stroke="none" stroke-width="1.3"/>
<text x="32" y="34" font-family="Arial, sans-serif" font-size="13" font-weight="700" fill="#c4b5fd" text-anchor="start" letter-spacing="2">SOLO / INCIDENT RESPONSE</text>
<text x="32" y="74" font-family="Arial, sans-serif" font-size="36" font-weight="700" fill="#ffffff" text-anchor="start">Layers of defence</text>
<text x="32" y="103" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#cbd5e1" text-anchor="start">Where the agent&#x27;s permissions are enforced</text>
<path d="M808,30 L794,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M808,30 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M822,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="808" cy="30" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="794" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="822" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="808" cy="58" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<text x="838" y="51" font-family="Arial, sans-serif" font-size="19" font-weight="700" fill="#ffffff" text-anchor="start" letter-spacing="1">SOLO.IO</text>
<rect x="842" y="76" width="86" height="28" rx="7" fill="#272c46" stroke="none" stroke-width="1.3"/>
<text x="885.0" y="95" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#d8ddec" text-anchor="middle">mesh1</text>
<g data-node="request">
<rect x="32" y="144" width="896" height="52" rx="12" fill="#ffffff" stroke="#dce1ed" stroke-width="1.3" data-node-box="true"/>
<circle cx="62" cy="161" r="6" fill="none" stroke="#58657c" stroke-width="1.8"/>
<path d="M51,184 L51,181 L55,174 L69,174 L73,181 L73,184" fill="none" stroke="#58657c" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="94" y="177" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Investigate the payments outage</text>
<rect x="720" y="156" width="192" height="28" rx="7" fill="#ede9fe" stroke="none" stroke-width="1.3"/>
<text x="816.0" y="175" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#5b21b6" text-anchor="middle">A2A via gateway</text>
</g>
<rect x="674" y="366" width="270" height="180" rx="16" fill="#edf4ff" stroke="#9dbdf6" stroke-width="1.3"/>
<rect x="685" y="350" width="247" height="30" rx="5" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<circle cx="700" cy="365" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="700" y="370" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">03</text>
<text x="722" y="371" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#1d4ed8" text-anchor="start" letter-spacing="0.3">WORKLOAD IDENTITY</text>
<path d="M284,302 L340,302" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="312" y="290" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="middle">calls</text>
<path d="M612,286 L680,286" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="272" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">LLM</text>
<path d="M612,452 L680,452" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="438" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">MCP</text>
<path d="M156,380 L156,418" fill="none" stroke="#776b9f" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="176" y="408" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="start">outside check</text>
<path d="M284,354 L314,354 L314,580 L638,580" fill="none" stroke="#be3455" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 6" data-connector="true"/>
<text x="362" y="567" font-family="Arial, sans-serif" font-size="18" font-weight="700" fill="#be3455" text-anchor="start">Direct MCP call</text>
<circle cx="653" cy="580" r="12" fill="#fff1f2" stroke="#be3455" stroke-width="2"/>
<path d="M648,575 L658,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M658,575 L648,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<text x="679" y="586" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#be3455" text-anchor="start" letter-spacing="0.5">REFUSED BY ISTIO</text>
<g data-node="agent">
<rect x="32" y="226" width="250" height="152" rx="12" fill="#ffffff" stroke="#dcd5ee" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="50" y="246" width="42" height="42" rx="10" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<rect x="57" y="257" width="28" height="22" rx="6" fill="none" stroke="#7c3aed" stroke-width="1.3"/>
<path d="M71,250 L71,257" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="71" cy="250" r="2.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="65" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="77" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<path d="M65,273 L77,273" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="105" y="275" font-family="Arial, sans-serif" font-size="29" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="52" y="314" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#172033" text-anchor="start">Incident-response agent</text>
<rect x="52" y="332" width="171" height="28" rx="7" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<text x="137.5" y="351" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#6d28d9" text-anchor="middle">Declarative Agent</text>
</g>
<g data-node="gateway">
<rect x="352" y="226" width="258" height="316" rx="12" fill="url(#defence-gateway-fill)" stroke="#5b4691" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M388,246 L404,252 L404,264 L400,273 L388,283 L376,273 L372,264 L372,252 L388,246" fill="none" stroke="#c4b5fd" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M382,263 L387,268 L395,257" fill="none" stroke="#ddd6fe" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/>
<text x="418" y="270" font-family="Arial, sans-serif" font-size="25" font-weight="700" fill="#ffffff" text-anchor="start">agentgateway</text>
<text x="374" y="294" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#c4b5fd" text-anchor="start">Policy enforcement</text>
<rect x="370" y="310" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="328" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="333" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">04</text>
<text x="414" y="335" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Caller identity</text>
<rect x="370" y="353" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="371" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="376" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">05</text>
<text x="414" y="378" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Tool permissions</text>
<rect x="370" y="396" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="414" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="419" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">06</text>
<text x="414" y="421" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Personal data</text>
<rect x="370" y="439" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="457" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="462" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">07</text>
<text x="414" y="464" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Request rate</text>
<path d="M374,491 L589,491" fill="none" stroke="#5a4b7c" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="391" cy="514" r="13" fill="#3e355f" stroke="none" stroke-width="1.5"/>
<text x="391" y="519" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#d5cce9" text-anchor="middle">08</text>
<text x="414" y="521" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#ded7ef" text-anchor="start">Logs + metrics</text>
</g>
<g data-node="claude">
<rect x="692" y="226" width="236" height="120" rx="12" fill="#fffaf5" stroke="#ead9cb" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M729.0,261.0 L740.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,263.5 L737.856406460551,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,265.3301270189222 L732.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,266.0 L724.0,277.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,265.3301270189222 L716.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,263.5 L710.143593539449,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.0,261.0 L708.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,258.5 L710.143593539449,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,256.6698729810778 L716.0,247.143593539449" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,256.0 L724.0,245.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,256.6698729810778 L732.0,247.14359353944897" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,258.5 L737.856406460551,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<text x="754" y="270" font-family="Arial, sans-serif" font-size="27" font-weight="700" fill="#643923" text-anchor="start">Claude</text>
<text x="712" y="310" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Haiku 4.5</text>
<text x="907" y="333" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#946e56" text-anchor="end">Anthropic</text>
</g>
<g data-node="tools">
<rect x="692" y="386" width="236" height="142" rx="12" fill="#ffffff" stroke="#c9d7ed" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="712" y="405" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="409" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="416" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="420" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="427" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="431" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<text x="754" y="429" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="712" y="467" font-family="Arial, sans-serif" font-size="23" font-weight="400" fill="#172033" text-anchor="start">Incident tools</text>
<text x="712" y="503" font-family="Menlo, Consolas, monospace" font-size="17" font-weight="400" fill="#58657c" text-anchor="start">MCPServer</text>
</g>
<g data-node="egress">
<rect x="32" y="430" width="250" height="112" rx="12" fill="#eff6ff" stroke="#c4d7f5" stroke-width="1.3" data-node-box="true"/>
<path d="M53,469 L65,445 L65,469 L53,469" fill="#60a5fa" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M69,450 L78,469 L69,469 L69,450" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M50,475 L80,475 L75,480 L55,480 L50,475" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="96" y="471" font-family="Arial, sans-serif" font-size="23" font-weight="700" fill="#1e40af" text-anchor="start">Istio egress</text>
<circle cx="253" cy="449" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="253" y="454" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">02</text>
<text x="52" y="515" font-family="Arial, sans-serif" font-size="19" font-weight="400" fill="#365a8c" text-anchor="start">example.com is refused</text>
</g>
<text x="32" y="631" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#172033" text-anchor="start">Ask the agent to close every incident</text>
<text x="32" y="658" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#58657c" text-anchor="start">Closing the record does not resolve the fault.</text>
<g data-node="before">
<rect x="32" y="680" width="436" height="166" rx="12" fill="#fff5f6" stroke="#f0bdc9" stroke-width="1.3" data-node-box="true"/>
<text x="52" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#a82e4b" text-anchor="start" letter-spacing="0.8">WITHOUT A TOOL POLICY</text>
<text x="52" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#172033" text-anchor="start">0</text>
<text x="85" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#58657c" text-anchor="start">open</text>
<path d="M216,727 L216,768" fill="none" stroke="#eccbd3" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="244" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#be3455" text-anchor="start">3</text>
<text x="279" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#be3455" text-anchor="start">closed</text>
<text x="52" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#9f2946" text-anchor="start">3 faults still unresolved</text>
<text x="52" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#9f6270" text-anchor="start">Bulk close succeeded</text>
</g>
<g data-node="after">
<rect x="492" y="680" width="436" height="166" rx="12" fill="#effbf5" stroke="#a6dac1" stroke-width="1.3" data-node-box="true"/>
<text x="512" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#087f5b" text-anchor="start" letter-spacing="0.8">WITH A READ-ONLY POLICY</text>
<text x="512" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#087f5b" text-anchor="start">3</text>
<text x="547" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#087f5b" text-anchor="start">open</text>
<rect x="741" y="731" width="166" height="28" rx="7" fill="#d0f0df" stroke="none" stroke-width="1.3"/>
<text x="824.0" y="750" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#086b4e" text-anchor="middle">WRITE REFUSED</text>
<text x="512" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#086b4e" text-anchor="start">Read-only tools still work</text>
<text x="512" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#46735f" text-anchor="start">Bulk close refused</text>
</g>
<text x="32" y="881" font-family="Arial, sans-serif" font-size="16" font-weight="400" fill="#58657c" text-anchor="start">Illustrated chapter-5 outcome. Live incident records and evidence are in the App tab.</text>
</svg>

### Without the control

Make an anonymous model request from the kagent runtime. This is a protocol check, separate from the agent's normal configured calls.

**Expected:** HTTP 200 without a token.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" -H 'Content-Type: application/json' -d '{"model":"claude-haiku-4-5","max_tokens":16,"messages":[{"role":"user","content":"Reply only READY"}]}'

### Add the control

Read and apply the public JWT policy. Require the lab issuer and audience on the gateway.

**Expected:** caller-identity is Accepted and Attached.

In [ ]:
cat "$DD_STATE/07-caller-identity.json"
kubectl --context kind-mesh1 apply -f "$DD_STATE/07-caller-identity.json"
dd_wait_policy caller-identity

### With the control

Repeat the anonymous request, then ask the kagent agent to investigate using its signed identity. Read the gateway's 401 log.

**Expected:** The anonymous request gets 401. The agent's authenticated model and tool calls still complete.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec deploy/defence-agent -- curl -sS --max-time 30 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" -H 'Content-Type: application/json' -d '{"model":"claude-haiku-4-5","max_tokens":16,"messages":[{"role":"user","content":"Reply only READY"}]}'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
kubectl --context kind-mesh1 -n dd-gateway logs deploy/dd-gateway --since=30s | grep 'http.status=401'

## 5. Stop an agent closing unresolved incidents

Ask the agent to close every incident. Without a tool policy, all three records change to closed even though their faults remain unresolved. Restore the sample records, restrict Alice to investigation tools, then repeat the request.

A prompt is not the permission boundary. The gateway filters tools/list and refuses tools/call independently of the agent's behaviour.

<svg id="defence-architecture" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 900" width="100%" role="img" aria-labelledby="defence-title defence-desc">
<title id="defence-title">Layers of defence for a kagent incident-response agent</title>
<desc id="defence-desc">kagent runs the agent and incident MCP tools. Agentgateway enforces caller identity, tool permissions, prompt guards and rate limits. Istio refuses direct tool access and example.com egress. Before the tool policy, three unresolved incidents are closed. After the policy, the bulk close is refused and investigation remains available.</desc>
<defs>
      <linearGradient id="defence-gateway-fill" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#382763"/><stop offset="1" stop-color="#231c40"/></linearGradient>
      <filter id="defence-shadow" x="-15%" y="-15%" width="130%" height="140%"><feDropShadow dx="0" dy="5" stdDeviation="6" flood-color="#232142" flood-opacity=".09"/></filter>
      <marker id="defence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M1 1 L9 5 L1 9 Z" fill="#776b9f"/></marker>
    </defs>
<rect x="0" y="0" width="960" height="900" rx="20" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<rect x="0" y="0" width="960" height="120" rx="20" fill="#101426" stroke="none" stroke-width="1.3"/>
<rect x="0" y="90" width="960" height="30" rx="0" fill="#101426" stroke="none" stroke-width="1.3"/>
<text x="32" y="34" font-family="Arial, sans-serif" font-size="13" font-weight="700" fill="#c4b5fd" text-anchor="start" letter-spacing="2">SOLO / INCIDENT RESPONSE</text>
<text x="32" y="74" font-family="Arial, sans-serif" font-size="36" font-weight="700" fill="#ffffff" text-anchor="start">Layers of defence</text>
<text x="32" y="103" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#cbd5e1" text-anchor="start">Where the agent&#x27;s permissions are enforced</text>
<path d="M808,30 L794,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M808,30 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M822,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="808" cy="30" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="794" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="822" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="808" cy="58" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<text x="838" y="51" font-family="Arial, sans-serif" font-size="19" font-weight="700" fill="#ffffff" text-anchor="start" letter-spacing="1">SOLO.IO</text>
<rect x="842" y="76" width="86" height="28" rx="7" fill="#272c46" stroke="none" stroke-width="1.3"/>
<text x="885.0" y="95" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#d8ddec" text-anchor="middle">mesh1</text>
<g data-node="request">
<rect x="32" y="144" width="896" height="52" rx="12" fill="#ffffff" stroke="#dce1ed" stroke-width="1.3" data-node-box="true"/>
<circle cx="62" cy="161" r="6" fill="none" stroke="#58657c" stroke-width="1.8"/>
<path d="M51,184 L51,181 L55,174 L69,174 L73,181 L73,184" fill="none" stroke="#58657c" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="94" y="177" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Investigate the payments outage</text>
<rect x="720" y="156" width="192" height="28" rx="7" fill="#ede9fe" stroke="none" stroke-width="1.3"/>
<text x="816.0" y="175" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#5b21b6" text-anchor="middle">A2A via gateway</text>
</g>
<rect x="674" y="366" width="270" height="180" rx="16" fill="#edf4ff" stroke="#9dbdf6" stroke-width="1.3"/>
<rect x="685" y="350" width="247" height="30" rx="5" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<circle cx="700" cy="365" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="700" y="370" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">03</text>
<text x="722" y="371" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#1d4ed8" text-anchor="start" letter-spacing="0.3">WORKLOAD IDENTITY</text>
<path d="M284,302 L340,302" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="312" y="290" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="middle">calls</text>
<path d="M612,286 L680,286" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="272" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">LLM</text>
<path d="M612,452 L680,452" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="438" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">MCP</text>
<path d="M156,380 L156,418" fill="none" stroke="#776b9f" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="176" y="408" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="start">outside check</text>
<path d="M284,354 L314,354 L314,580 L638,580" fill="none" stroke="#be3455" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 6" data-connector="true"/>
<text x="362" y="567" font-family="Arial, sans-serif" font-size="18" font-weight="700" fill="#be3455" text-anchor="start">Direct MCP call</text>
<circle cx="653" cy="580" r="12" fill="#fff1f2" stroke="#be3455" stroke-width="2"/>
<path d="M648,575 L658,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M658,575 L648,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<text x="679" y="586" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#be3455" text-anchor="start" letter-spacing="0.5">REFUSED BY ISTIO</text>
<g data-node="agent">
<rect x="32" y="226" width="250" height="152" rx="12" fill="#ffffff" stroke="#dcd5ee" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="50" y="246" width="42" height="42" rx="10" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<rect x="57" y="257" width="28" height="22" rx="6" fill="none" stroke="#7c3aed" stroke-width="1.3"/>
<path d="M71,250 L71,257" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="71" cy="250" r="2.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="65" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="77" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<path d="M65,273 L77,273" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="105" y="275" font-family="Arial, sans-serif" font-size="29" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="52" y="314" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#172033" text-anchor="start">Incident-response agent</text>
<rect x="52" y="332" width="171" height="28" rx="7" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<text x="137.5" y="351" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#6d28d9" text-anchor="middle">Declarative Agent</text>
</g>
<g data-node="gateway">
<rect x="352" y="226" width="258" height="316" rx="12" fill="url(#defence-gateway-fill)" stroke="#5b4691" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M388,246 L404,252 L404,264 L400,273 L388,283 L376,273 L372,264 L372,252 L388,246" fill="none" stroke="#c4b5fd" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M382,263 L387,268 L395,257" fill="none" stroke="#ddd6fe" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/>
<text x="418" y="270" font-family="Arial, sans-serif" font-size="25" font-weight="700" fill="#ffffff" text-anchor="start">agentgateway</text>
<text x="374" y="294" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#c4b5fd" text-anchor="start">Policy enforcement</text>
<rect x="370" y="310" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="328" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="333" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">04</text>
<text x="414" y="335" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Caller identity</text>
<rect x="370" y="353" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="371" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="376" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">05</text>
<text x="414" y="378" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Tool permissions</text>
<rect x="370" y="396" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="414" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="419" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">06</text>
<text x="414" y="421" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Personal data</text>
<rect x="370" y="439" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="457" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="462" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">07</text>
<text x="414" y="464" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Request rate</text>
<path d="M374,491 L589,491" fill="none" stroke="#5a4b7c" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="391" cy="514" r="13" fill="#3e355f" stroke="none" stroke-width="1.5"/>
<text x="391" y="519" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#d5cce9" text-anchor="middle">08</text>
<text x="414" y="521" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#ded7ef" text-anchor="start">Logs + metrics</text>
</g>
<g data-node="claude">
<rect x="692" y="226" width="236" height="120" rx="12" fill="#fffaf5" stroke="#ead9cb" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M729.0,261.0 L740.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,263.5 L737.856406460551,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,265.3301270189222 L732.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,266.0 L724.0,277.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,265.3301270189222 L716.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,263.5 L710.143593539449,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.0,261.0 L708.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,258.5 L710.143593539449,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,256.6698729810778 L716.0,247.143593539449" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,256.0 L724.0,245.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,256.6698729810778 L732.0,247.14359353944897" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,258.5 L737.856406460551,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<text x="754" y="270" font-family="Arial, sans-serif" font-size="27" font-weight="700" fill="#643923" text-anchor="start">Claude</text>
<text x="712" y="310" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Haiku 4.5</text>
<text x="907" y="333" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#946e56" text-anchor="end">Anthropic</text>
</g>
<g data-node="tools">
<rect x="692" y="386" width="236" height="142" rx="12" fill="#ffffff" stroke="#c9d7ed" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="712" y="405" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="409" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="416" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="420" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="427" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="431" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<text x="754" y="429" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="712" y="467" font-family="Arial, sans-serif" font-size="23" font-weight="400" fill="#172033" text-anchor="start">Incident tools</text>
<text x="712" y="503" font-family="Menlo, Consolas, monospace" font-size="17" font-weight="400" fill="#58657c" text-anchor="start">MCPServer</text>
</g>
<g data-node="egress">
<rect x="32" y="430" width="250" height="112" rx="12" fill="#eff6ff" stroke="#c4d7f5" stroke-width="1.3" data-node-box="true"/>
<path d="M53,469 L65,445 L65,469 L53,469" fill="#60a5fa" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M69,450 L78,469 L69,469 L69,450" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M50,475 L80,475 L75,480 L55,480 L50,475" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="96" y="471" font-family="Arial, sans-serif" font-size="23" font-weight="700" fill="#1e40af" text-anchor="start">Istio egress</text>
<circle cx="253" cy="449" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="253" y="454" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">02</text>
<text x="52" y="515" font-family="Arial, sans-serif" font-size="19" font-weight="400" fill="#365a8c" text-anchor="start">example.com is refused</text>
</g>
<text x="32" y="631" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#172033" text-anchor="start">Ask the agent to close every incident</text>
<text x="32" y="658" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#58657c" text-anchor="start">Closing the record does not resolve the fault.</text>
<g data-node="before">
<rect x="32" y="680" width="436" height="166" rx="12" fill="#fff5f6" stroke="#f0bdc9" stroke-width="1.3" data-node-box="true"/>
<text x="52" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#a82e4b" text-anchor="start" letter-spacing="0.8">WITHOUT A TOOL POLICY</text>
<text x="52" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#172033" text-anchor="start">0</text>
<text x="85" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#58657c" text-anchor="start">open</text>
<path d="M216,727 L216,768" fill="none" stroke="#eccbd3" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="244" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#be3455" text-anchor="start">3</text>
<text x="279" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#be3455" text-anchor="start">closed</text>
<text x="52" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#9f2946" text-anchor="start">3 faults still unresolved</text>
<text x="52" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#9f6270" text-anchor="start">Bulk close succeeded</text>
</g>
<g data-node="after">
<rect x="492" y="680" width="436" height="166" rx="12" fill="#effbf5" stroke="#a6dac1" stroke-width="1.3" data-node-box="true"/>
<text x="512" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#087f5b" text-anchor="start" letter-spacing="0.8">WITH A READ-ONLY POLICY</text>
<text x="512" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#087f5b" text-anchor="start">3</text>
<text x="547" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#087f5b" text-anchor="start">open</text>
<rect x="741" y="731" width="166" height="28" rx="7" fill="#d0f0df" stroke="none" stroke-width="1.3"/>
<text x="824.0" y="750" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#086b4e" text-anchor="middle">WRITE REFUSED</text>
<text x="512" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#086b4e" text-anchor="start">Read-only tools still work</text>
<text x="512" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#46735f" text-anchor="start">Bulk close refused</text>
</g>
<text x="32" y="881" font-family="Arial, sans-serif" font-size="16" font-weight="400" fill="#58657c" text-anchor="start">Illustrated chapter-5 outcome. Live incident records and evidence are in the App tab.</text>
</svg>

### Without the control

Ask the agent to close every open incident. Watch the App tab: the open count falls to zero, but the unresolved count stays at three.

**Expected:** The real tool closes all three records. No underlying fault has been resolved.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use close_all_incidents to close every open incident now. Report the actual changed count.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

### Add the control

Permit only list_incidents and get_incident for Alice. As the administrator, restore the disposable records so the after check starts from the same state.

**Expected:** The policy is attached and the administrator restores three open incidents.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: tool-permissions, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: tools}
  backend:
    mcp:
      authorization:
        action: Allow
        policy:
          matchExpressions:
          - 'jwt.group == "admin" || mcp.tool.name in ["list_incidents", "get_incident"]'
YAML
dd_wait_policy tool-permissions
dd_mcp "$ADMIN" tools/call '{"name":"reset_demo_records","arguments":{}}'

### With the control

Repeat the same request to the agent. Check that write tools have disappeared, explicitly attempt bulk close, then ask the agent to read the records.

**Expected:** Bulk close is unavailable to the agent and the gateway refuses an explicit call. Investigation still works, and all three incidents remain open.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use close_all_incidents to close every open incident now. Report the actual changed count.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
printf 'Reader tools\n'; dd_mcp "$ALICE" tools/list
printf 'Reader call\n'; dd_mcp "$ALICE" tools/call '{"name":"close_all_incidents","arguments":{}}'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Use list_incidents to read the current incidents. Report the open count and affected services.' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
kubectl --context kind-mesh1 -n dd-gateway logs deploy/dd-gateway --since=30s | grep 'Unknown tool: close_all_incidents'

## 6. Refuse personal data before the model call

Ask the agent to repeat a specimen email address. Add a request guard that rejects email addresses, then send the same message again.

This uses a real model. The proof is a failed agent turn plus the gateway's 403 and Reject counter, not an echo simulator claiming what the model received.

<svg id="defence-architecture" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 900" width="100%" role="img" aria-labelledby="defence-title defence-desc">
<title id="defence-title">Layers of defence for a kagent incident-response agent</title>
<desc id="defence-desc">kagent runs the agent and incident MCP tools. Agentgateway enforces caller identity, tool permissions, prompt guards and rate limits. Istio refuses direct tool access and example.com egress. Before the tool policy, three unresolved incidents are closed. After the policy, the bulk close is refused and investigation remains available.</desc>
<defs>
      <linearGradient id="defence-gateway-fill" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#382763"/><stop offset="1" stop-color="#231c40"/></linearGradient>
      <filter id="defence-shadow" x="-15%" y="-15%" width="130%" height="140%"><feDropShadow dx="0" dy="5" stdDeviation="6" flood-color="#232142" flood-opacity=".09"/></filter>
      <marker id="defence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M1 1 L9 5 L1 9 Z" fill="#776b9f"/></marker>
    </defs>
<rect x="0" y="0" width="960" height="900" rx="20" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<rect x="0" y="0" width="960" height="120" rx="20" fill="#101426" stroke="none" stroke-width="1.3"/>
<rect x="0" y="90" width="960" height="30" rx="0" fill="#101426" stroke="none" stroke-width="1.3"/>
<text x="32" y="34" font-family="Arial, sans-serif" font-size="13" font-weight="700" fill="#c4b5fd" text-anchor="start" letter-spacing="2">SOLO / INCIDENT RESPONSE</text>
<text x="32" y="74" font-family="Arial, sans-serif" font-size="36" font-weight="700" fill="#ffffff" text-anchor="start">Layers of defence</text>
<text x="32" y="103" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#cbd5e1" text-anchor="start">Where the agent&#x27;s permissions are enforced</text>
<path d="M808,30 L794,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M808,30 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M822,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="808" cy="30" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="794" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="822" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="808" cy="58" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<text x="838" y="51" font-family="Arial, sans-serif" font-size="19" font-weight="700" fill="#ffffff" text-anchor="start" letter-spacing="1">SOLO.IO</text>
<rect x="842" y="76" width="86" height="28" rx="7" fill="#272c46" stroke="none" stroke-width="1.3"/>
<text x="885.0" y="95" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#d8ddec" text-anchor="middle">mesh1</text>
<g data-node="request">
<rect x="32" y="144" width="896" height="52" rx="12" fill="#ffffff" stroke="#dce1ed" stroke-width="1.3" data-node-box="true"/>
<circle cx="62" cy="161" r="6" fill="none" stroke="#58657c" stroke-width="1.8"/>
<path d="M51,184 L51,181 L55,174 L69,174 L73,181 L73,184" fill="none" stroke="#58657c" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="94" y="177" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Investigate the payments outage</text>
<rect x="720" y="156" width="192" height="28" rx="7" fill="#ede9fe" stroke="none" stroke-width="1.3"/>
<text x="816.0" y="175" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#5b21b6" text-anchor="middle">A2A via gateway</text>
</g>
<rect x="674" y="366" width="270" height="180" rx="16" fill="#edf4ff" stroke="#9dbdf6" stroke-width="1.3"/>
<rect x="685" y="350" width="247" height="30" rx="5" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<circle cx="700" cy="365" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="700" y="370" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">03</text>
<text x="722" y="371" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#1d4ed8" text-anchor="start" letter-spacing="0.3">WORKLOAD IDENTITY</text>
<path d="M284,302 L340,302" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="312" y="290" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="middle">calls</text>
<path d="M612,286 L680,286" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="272" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">LLM</text>
<path d="M612,452 L680,452" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="438" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">MCP</text>
<path d="M156,380 L156,418" fill="none" stroke="#776b9f" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="176" y="408" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="start">outside check</text>
<path d="M284,354 L314,354 L314,580 L638,580" fill="none" stroke="#be3455" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 6" data-connector="true"/>
<text x="362" y="567" font-family="Arial, sans-serif" font-size="18" font-weight="700" fill="#be3455" text-anchor="start">Direct MCP call</text>
<circle cx="653" cy="580" r="12" fill="#fff1f2" stroke="#be3455" stroke-width="2"/>
<path d="M648,575 L658,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M658,575 L648,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<text x="679" y="586" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#be3455" text-anchor="start" letter-spacing="0.5">REFUSED BY ISTIO</text>
<g data-node="agent">
<rect x="32" y="226" width="250" height="152" rx="12" fill="#ffffff" stroke="#dcd5ee" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="50" y="246" width="42" height="42" rx="10" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<rect x="57" y="257" width="28" height="22" rx="6" fill="none" stroke="#7c3aed" stroke-width="1.3"/>
<path d="M71,250 L71,257" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="71" cy="250" r="2.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="65" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="77" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<path d="M65,273 L77,273" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="105" y="275" font-family="Arial, sans-serif" font-size="29" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="52" y="314" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#172033" text-anchor="start">Incident-response agent</text>
<rect x="52" y="332" width="171" height="28" rx="7" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<text x="137.5" y="351" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#6d28d9" text-anchor="middle">Declarative Agent</text>
</g>
<g data-node="gateway">
<rect x="352" y="226" width="258" height="316" rx="12" fill="url(#defence-gateway-fill)" stroke="#5b4691" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M388,246 L404,252 L404,264 L400,273 L388,283 L376,273 L372,264 L372,252 L388,246" fill="none" stroke="#c4b5fd" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M382,263 L387,268 L395,257" fill="none" stroke="#ddd6fe" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/>
<text x="418" y="270" font-family="Arial, sans-serif" font-size="25" font-weight="700" fill="#ffffff" text-anchor="start">agentgateway</text>
<text x="374" y="294" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#c4b5fd" text-anchor="start">Policy enforcement</text>
<rect x="370" y="310" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="328" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="333" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">04</text>
<text x="414" y="335" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Caller identity</text>
<rect x="370" y="353" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="371" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="376" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">05</text>
<text x="414" y="378" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Tool permissions</text>
<rect x="370" y="396" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="414" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="419" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">06</text>
<text x="414" y="421" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Personal data</text>
<rect x="370" y="439" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="457" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="462" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">07</text>
<text x="414" y="464" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Request rate</text>
<path d="M374,491 L589,491" fill="none" stroke="#5a4b7c" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="391" cy="514" r="13" fill="#3e355f" stroke="none" stroke-width="1.5"/>
<text x="391" y="519" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#d5cce9" text-anchor="middle">08</text>
<text x="414" y="521" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#ded7ef" text-anchor="start">Logs + metrics</text>
</g>
<g data-node="claude">
<rect x="692" y="226" width="236" height="120" rx="12" fill="#fffaf5" stroke="#ead9cb" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M729.0,261.0 L740.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,263.5 L737.856406460551,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,265.3301270189222 L732.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,266.0 L724.0,277.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,265.3301270189222 L716.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,263.5 L710.143593539449,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.0,261.0 L708.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,258.5 L710.143593539449,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,256.6698729810778 L716.0,247.143593539449" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,256.0 L724.0,245.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,256.6698729810778 L732.0,247.14359353944897" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,258.5 L737.856406460551,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<text x="754" y="270" font-family="Arial, sans-serif" font-size="27" font-weight="700" fill="#643923" text-anchor="start">Claude</text>
<text x="712" y="310" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Haiku 4.5</text>
<text x="907" y="333" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#946e56" text-anchor="end">Anthropic</text>
</g>
<g data-node="tools">
<rect x="692" y="386" width="236" height="142" rx="12" fill="#ffffff" stroke="#c9d7ed" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="712" y="405" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="409" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="416" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="420" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="427" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="431" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<text x="754" y="429" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="712" y="467" font-family="Arial, sans-serif" font-size="23" font-weight="400" fill="#172033" text-anchor="start">Incident tools</text>
<text x="712" y="503" font-family="Menlo, Consolas, monospace" font-size="17" font-weight="400" fill="#58657c" text-anchor="start">MCPServer</text>
</g>
<g data-node="egress">
<rect x="32" y="430" width="250" height="112" rx="12" fill="#eff6ff" stroke="#c4d7f5" stroke-width="1.3" data-node-box="true"/>
<path d="M53,469 L65,445 L65,469 L53,469" fill="#60a5fa" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M69,450 L78,469 L69,469 L69,450" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M50,475 L80,475 L75,480 L55,480 L50,475" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="96" y="471" font-family="Arial, sans-serif" font-size="23" font-weight="700" fill="#1e40af" text-anchor="start">Istio egress</text>
<circle cx="253" cy="449" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="253" y="454" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">02</text>
<text x="52" y="515" font-family="Arial, sans-serif" font-size="19" font-weight="400" fill="#365a8c" text-anchor="start">example.com is refused</text>
</g>
<text x="32" y="631" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#172033" text-anchor="start">Ask the agent to close every incident</text>
<text x="32" y="658" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#58657c" text-anchor="start">Closing the record does not resolve the fault.</text>
<g data-node="before">
<rect x="32" y="680" width="436" height="166" rx="12" fill="#fff5f6" stroke="#f0bdc9" stroke-width="1.3" data-node-box="true"/>
<text x="52" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#a82e4b" text-anchor="start" letter-spacing="0.8">WITHOUT A TOOL POLICY</text>
<text x="52" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#172033" text-anchor="start">0</text>
<text x="85" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#58657c" text-anchor="start">open</text>
<path d="M216,727 L216,768" fill="none" stroke="#eccbd3" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="244" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#be3455" text-anchor="start">3</text>
<text x="279" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#be3455" text-anchor="start">closed</text>
<text x="52" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#9f2946" text-anchor="start">3 faults still unresolved</text>
<text x="52" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#9f6270" text-anchor="start">Bulk close succeeded</text>
</g>
<g data-node="after">
<rect x="492" y="680" width="436" height="166" rx="12" fill="#effbf5" stroke="#a6dac1" stroke-width="1.3" data-node-box="true"/>
<text x="512" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#087f5b" text-anchor="start" letter-spacing="0.8">WITH A READ-ONLY POLICY</text>
<text x="512" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#087f5b" text-anchor="start">3</text>
<text x="547" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#087f5b" text-anchor="start">open</text>
<rect x="741" y="731" width="166" height="28" rx="7" fill="#d0f0df" stroke="none" stroke-width="1.3"/>
<text x="824.0" y="750" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#086b4e" text-anchor="middle">WRITE REFUSED</text>
<text x="512" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#086b4e" text-anchor="start">Read-only tools still work</text>
<text x="512" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#46735f" text-anchor="start">Bulk close refused</text>
</g>
<text x="32" y="881" font-family="Arial, sans-serif" font-size="16" font-weight="400" fill="#58657c" text-anchor="start">Illustrated chapter-5 outcome. Live incident records and evidence are in the App tab.</text>
</svg>

### Without the control

Ask the agent to repeat the specimen email.

**Expected:** The agent completes and includes alice@example.test in its reply.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Repeat this sample email exactly: alice@example.test' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

### Add the control

Reject recognised email addresses on the model route before the provider call.

**Expected:** reject-personal-data is Accepted and Attached.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: reject-personal-data, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, name: model}
  backend:
    ai:
      promptGuard:
        request:
        - regex:
            action: Reject
            builtins: [Email]
          response:
            statusCode: 403
            message: "Personal data refused before the model call"
YAML
dd_wait_policy reject-personal-data

### With the control

Send the same message through the agent. Read the gateway's guardrail evidence, then send a message without personal data.

**Expected:** The email turn fails with a gateway 403 and a Reject counter. The clean turn completes.

In [ ]:
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Repeat this sample email exactly: alice@example.test' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}' 
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'
DD_PUBLIC="http://$(kubectl --context kind-mesh1 -n dd-gateway get gateway dd-gateway -o jsonpath='{.status.addresses[0].value}')"
BODY=$(jq -nc --arg prompt 'Reply only READY' '{jsonrpc:"2.0",id:"lab",method:"message/send",params:{message:{role:"user",messageId:"defence-check",parts:[{kind:"text",text:$prompt}]}}}')
curl -fsS --max-time 120 "$DD_PUBLIC/a2a/" -H "Authorization: Bearer $ALICE" -H 'Content-Type: application/json' -d "$BODY" | tee "$DD_STATE/last-agent-task.json" | jq '{state:.result.status.state,calls:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_call")|.data|{name,args}],results:[.result.history[]?.parts[]?|select(.metadata.kagent_type=="function_response")|.data|{tool:.name,output:(.response.structuredContent // (try (.response.content[0].text|fromjson) catch null) // .response.content),isError:(.response.isError // false)}],answer:[.result.artifacts[]?.parts[]?.text],error:.error}'

## 7. Limit model requests per caller

Send a bounded sequence of model requests from the kagent runtime. Add a global limit of three requests per minute per signed subject, then repeat the check.

This is a protocol-level rate check. Agent turns can make multiple model requests, so a request allowance is not a turn allowance. Replies are capped at sixteen tokens and use the real provider.

<svg id="defence-architecture" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 900" width="100%" role="img" aria-labelledby="defence-title defence-desc">
<title id="defence-title">Layers of defence for a kagent incident-response agent</title>
<desc id="defence-desc">kagent runs the agent and incident MCP tools. Agentgateway enforces caller identity, tool permissions, prompt guards and rate limits. Istio refuses direct tool access and example.com egress. Before the tool policy, three unresolved incidents are closed. After the policy, the bulk close is refused and investigation remains available.</desc>
<defs>
      <linearGradient id="defence-gateway-fill" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#382763"/><stop offset="1" stop-color="#231c40"/></linearGradient>
      <filter id="defence-shadow" x="-15%" y="-15%" width="130%" height="140%"><feDropShadow dx="0" dy="5" stdDeviation="6" flood-color="#232142" flood-opacity=".09"/></filter>
      <marker id="defence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M1 1 L9 5 L1 9 Z" fill="#776b9f"/></marker>
    </defs>
<rect x="0" y="0" width="960" height="900" rx="20" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<rect x="0" y="0" width="960" height="120" rx="20" fill="#101426" stroke="none" stroke-width="1.3"/>
<rect x="0" y="90" width="960" height="30" rx="0" fill="#101426" stroke="none" stroke-width="1.3"/>
<text x="32" y="34" font-family="Arial, sans-serif" font-size="13" font-weight="700" fill="#c4b5fd" text-anchor="start" letter-spacing="2">SOLO / INCIDENT RESPONSE</text>
<text x="32" y="74" font-family="Arial, sans-serif" font-size="36" font-weight="700" fill="#ffffff" text-anchor="start">Layers of defence</text>
<text x="32" y="103" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#cbd5e1" text-anchor="start">Where the agent&#x27;s permissions are enforced</text>
<path d="M808,30 L794,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M808,30 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M822,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="808" cy="30" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="794" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="822" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="808" cy="58" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<text x="838" y="51" font-family="Arial, sans-serif" font-size="19" font-weight="700" fill="#ffffff" text-anchor="start" letter-spacing="1">SOLO.IO</text>
<rect x="842" y="76" width="86" height="28" rx="7" fill="#272c46" stroke="none" stroke-width="1.3"/>
<text x="885.0" y="95" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#d8ddec" text-anchor="middle">mesh1</text>
<g data-node="request">
<rect x="32" y="144" width="896" height="52" rx="12" fill="#ffffff" stroke="#dce1ed" stroke-width="1.3" data-node-box="true"/>
<circle cx="62" cy="161" r="6" fill="none" stroke="#58657c" stroke-width="1.8"/>
<path d="M51,184 L51,181 L55,174 L69,174 L73,181 L73,184" fill="none" stroke="#58657c" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="94" y="177" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Investigate the payments outage</text>
<rect x="720" y="156" width="192" height="28" rx="7" fill="#ede9fe" stroke="none" stroke-width="1.3"/>
<text x="816.0" y="175" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#5b21b6" text-anchor="middle">A2A via gateway</text>
</g>
<rect x="674" y="366" width="270" height="180" rx="16" fill="#edf4ff" stroke="#9dbdf6" stroke-width="1.3"/>
<rect x="685" y="350" width="247" height="30" rx="5" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<circle cx="700" cy="365" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="700" y="370" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">03</text>
<text x="722" y="371" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#1d4ed8" text-anchor="start" letter-spacing="0.3">WORKLOAD IDENTITY</text>
<path d="M284,302 L340,302" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="312" y="290" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="middle">calls</text>
<path d="M612,286 L680,286" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="272" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">LLM</text>
<path d="M612,452 L680,452" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="438" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">MCP</text>
<path d="M156,380 L156,418" fill="none" stroke="#776b9f" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="176" y="408" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="start">outside check</text>
<path d="M284,354 L314,354 L314,580 L638,580" fill="none" stroke="#be3455" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 6" data-connector="true"/>
<text x="362" y="567" font-family="Arial, sans-serif" font-size="18" font-weight="700" fill="#be3455" text-anchor="start">Direct MCP call</text>
<circle cx="653" cy="580" r="12" fill="#fff1f2" stroke="#be3455" stroke-width="2"/>
<path d="M648,575 L658,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M658,575 L648,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<text x="679" y="586" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#be3455" text-anchor="start" letter-spacing="0.5">REFUSED BY ISTIO</text>
<g data-node="agent">
<rect x="32" y="226" width="250" height="152" rx="12" fill="#ffffff" stroke="#dcd5ee" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="50" y="246" width="42" height="42" rx="10" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<rect x="57" y="257" width="28" height="22" rx="6" fill="none" stroke="#7c3aed" stroke-width="1.3"/>
<path d="M71,250 L71,257" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="71" cy="250" r="2.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="65" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="77" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<path d="M65,273 L77,273" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="105" y="275" font-family="Arial, sans-serif" font-size="29" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="52" y="314" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#172033" text-anchor="start">Incident-response agent</text>
<rect x="52" y="332" width="171" height="28" rx="7" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<text x="137.5" y="351" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#6d28d9" text-anchor="middle">Declarative Agent</text>
</g>
<g data-node="gateway">
<rect x="352" y="226" width="258" height="316" rx="12" fill="url(#defence-gateway-fill)" stroke="#5b4691" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M388,246 L404,252 L404,264 L400,273 L388,283 L376,273 L372,264 L372,252 L388,246" fill="none" stroke="#c4b5fd" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M382,263 L387,268 L395,257" fill="none" stroke="#ddd6fe" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/>
<text x="418" y="270" font-family="Arial, sans-serif" font-size="25" font-weight="700" fill="#ffffff" text-anchor="start">agentgateway</text>
<text x="374" y="294" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#c4b5fd" text-anchor="start">Policy enforcement</text>
<rect x="370" y="310" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="328" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="333" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">04</text>
<text x="414" y="335" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Caller identity</text>
<rect x="370" y="353" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="371" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="376" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">05</text>
<text x="414" y="378" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Tool permissions</text>
<rect x="370" y="396" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="414" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="419" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">06</text>
<text x="414" y="421" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Personal data</text>
<rect x="370" y="439" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="457" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="462" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">07</text>
<text x="414" y="464" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Request rate</text>
<path d="M374,491 L589,491" fill="none" stroke="#5a4b7c" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="391" cy="514" r="13" fill="#3e355f" stroke="none" stroke-width="1.5"/>
<text x="391" y="519" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#d5cce9" text-anchor="middle">08</text>
<text x="414" y="521" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#ded7ef" text-anchor="start">Logs + metrics</text>
</g>
<g data-node="claude">
<rect x="692" y="226" width="236" height="120" rx="12" fill="#fffaf5" stroke="#ead9cb" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M729.0,261.0 L740.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,263.5 L737.856406460551,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,265.3301270189222 L732.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,266.0 L724.0,277.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,265.3301270189222 L716.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,263.5 L710.143593539449,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.0,261.0 L708.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,258.5 L710.143593539449,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,256.6698729810778 L716.0,247.143593539449" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,256.0 L724.0,245.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,256.6698729810778 L732.0,247.14359353944897" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,258.5 L737.856406460551,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<text x="754" y="270" font-family="Arial, sans-serif" font-size="27" font-weight="700" fill="#643923" text-anchor="start">Claude</text>
<text x="712" y="310" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Haiku 4.5</text>
<text x="907" y="333" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#946e56" text-anchor="end">Anthropic</text>
</g>
<g data-node="tools">
<rect x="692" y="386" width="236" height="142" rx="12" fill="#ffffff" stroke="#c9d7ed" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="712" y="405" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="409" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="416" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="420" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="427" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="431" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<text x="754" y="429" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="712" y="467" font-family="Arial, sans-serif" font-size="23" font-weight="400" fill="#172033" text-anchor="start">Incident tools</text>
<text x="712" y="503" font-family="Menlo, Consolas, monospace" font-size="17" font-weight="400" fill="#58657c" text-anchor="start">MCPServer</text>
</g>
<g data-node="egress">
<rect x="32" y="430" width="250" height="112" rx="12" fill="#eff6ff" stroke="#c4d7f5" stroke-width="1.3" data-node-box="true"/>
<path d="M53,469 L65,445 L65,469 L53,469" fill="#60a5fa" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M69,450 L78,469 L69,469 L69,450" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M50,475 L80,475 L75,480 L55,480 L50,475" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="96" y="471" font-family="Arial, sans-serif" font-size="23" font-weight="700" fill="#1e40af" text-anchor="start">Istio egress</text>
<circle cx="253" cy="449" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="253" y="454" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">02</text>
<text x="52" y="515" font-family="Arial, sans-serif" font-size="19" font-weight="400" fill="#365a8c" text-anchor="start">example.com is refused</text>
</g>
<text x="32" y="631" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#172033" text-anchor="start">Ask the agent to close every incident</text>
<text x="32" y="658" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#58657c" text-anchor="start">Closing the record does not resolve the fault.</text>
<g data-node="before">
<rect x="32" y="680" width="436" height="166" rx="12" fill="#fff5f6" stroke="#f0bdc9" stroke-width="1.3" data-node-box="true"/>
<text x="52" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#a82e4b" text-anchor="start" letter-spacing="0.8">WITHOUT A TOOL POLICY</text>
<text x="52" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#172033" text-anchor="start">0</text>
<text x="85" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#58657c" text-anchor="start">open</text>
<path d="M216,727 L216,768" fill="none" stroke="#eccbd3" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="244" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#be3455" text-anchor="start">3</text>
<text x="279" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#be3455" text-anchor="start">closed</text>
<text x="52" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#9f2946" text-anchor="start">3 faults still unresolved</text>
<text x="52" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#9f6270" text-anchor="start">Bulk close succeeded</text>
</g>
<g data-node="after">
<rect x="492" y="680" width="436" height="166" rx="12" fill="#effbf5" stroke="#a6dac1" stroke-width="1.3" data-node-box="true"/>
<text x="512" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#087f5b" text-anchor="start" letter-spacing="0.8">WITH A READ-ONLY POLICY</text>
<text x="512" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#087f5b" text-anchor="start">3</text>
<text x="547" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#087f5b" text-anchor="start">open</text>
<rect x="741" y="731" width="166" height="28" rx="7" fill="#d0f0df" stroke="none" stroke-width="1.3"/>
<text x="824.0" y="750" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#086b4e" text-anchor="middle">WRITE REFUSED</text>
<text x="512" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#086b4e" text-anchor="start">Read-only tools still work</text>
<text x="512" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#46735f" text-anchor="start">Bulk close refused</text>
</g>
<text x="32" y="881" font-family="Arial, sans-serif" font-size="16" font-weight="400" fill="#58657c" text-anchor="start">Illustrated chapter-5 outcome. Live incident records and evidence are in the App tab.</text>
</svg>

### Without the control

Send the bounded rate check with fresh Alice and Bob subjects.

**Expected:** All requests return 200.

In [ ]:
dd_burst open "${N:-6}"

### Add the control

Apply the per-subject descriptor and wait for the limiter to load the current generation.

**Expected:** The policy is attached and the limiter reports ACCEPTED.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: ratelimit.solo.io/v1alpha1
kind: RateLimitConfig
metadata: {name: dd-per-person, namespace: dd-gateway}
spec:
  raw:
    descriptors:
    - key: dd-sub
      rateLimit: {requestsPerUnit: 3, unit: MINUTE}
    rateLimits:
    - actions:
      - cel: {expression: jwt.sub, key: dd-sub}
      type: REQUEST
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: per-person-rate, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, name: model}
  traffic:
    entRateLimit:
      global:
        rateLimitConfigRefs: [{name: dd-per-person}]
        backendRef:
          name: dd-rate-limiter
          namespace: agentgateway-system
          port: 8083
YAML
dd_wait_policy per-person-rate
dd_wait_rate

### With the control

Repeat the bounded check and read the 429 metric. A request as Bob checks that the allowance is per identity.

**Expected:** Alice reaches 429 while Bob still gets 200.

In [ ]:
dd_burst limited "${N:-6}"
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

## 8. Read the recorded outcomes by caller

The platform already records refusals. Add caller identity to the gateway metrics and generate another bounded rate check.

The live board shows the kagent Agent and its last A2A task alongside policy state, gateway counters and workload refusal evidence. Detection observes; it does not add another refusal control.

<svg id="defence-architecture" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 900" width="100%" role="img" aria-labelledby="defence-title defence-desc">
<title id="defence-title">Layers of defence for a kagent incident-response agent</title>
<desc id="defence-desc">kagent runs the agent and incident MCP tools. Agentgateway enforces caller identity, tool permissions, prompt guards and rate limits. Istio refuses direct tool access and example.com egress. Before the tool policy, three unresolved incidents are closed. After the policy, the bulk close is refused and investigation remains available.</desc>
<defs>
      <linearGradient id="defence-gateway-fill" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#382763"/><stop offset="1" stop-color="#231c40"/></linearGradient>
      <filter id="defence-shadow" x="-15%" y="-15%" width="130%" height="140%"><feDropShadow dx="0" dy="5" stdDeviation="6" flood-color="#232142" flood-opacity=".09"/></filter>
      <marker id="defence-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M1 1 L9 5 L1 9 Z" fill="#776b9f"/></marker>
    </defs>
<rect x="0" y="0" width="960" height="900" rx="20" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<rect x="0" y="0" width="960" height="120" rx="20" fill="#101426" stroke="none" stroke-width="1.3"/>
<rect x="0" y="90" width="960" height="30" rx="0" fill="#101426" stroke="none" stroke-width="1.3"/>
<text x="32" y="34" font-family="Arial, sans-serif" font-size="13" font-weight="700" fill="#c4b5fd" text-anchor="start" letter-spacing="2">SOLO / INCIDENT RESPONSE</text>
<text x="32" y="74" font-family="Arial, sans-serif" font-size="36" font-weight="700" fill="#ffffff" text-anchor="start">Layers of defence</text>
<text x="32" y="103" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#cbd5e1" text-anchor="start">Where the agent&#x27;s permissions are enforced</text>
<path d="M808,30 L794,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M808,30 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L822,44" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M794,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M822,44 L808,58" fill="none" stroke="#a78bfa" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="808" cy="30" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="794" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="822" cy="44" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<circle cx="808" cy="58" r="4" fill="#a78bfa" stroke="none" stroke-width="1.5"/>
<text x="838" y="51" font-family="Arial, sans-serif" font-size="19" font-weight="700" fill="#ffffff" text-anchor="start" letter-spacing="1">SOLO.IO</text>
<rect x="842" y="76" width="86" height="28" rx="7" fill="#272c46" stroke="none" stroke-width="1.3"/>
<text x="885.0" y="95" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#d8ddec" text-anchor="middle">mesh1</text>
<g data-node="request">
<rect x="32" y="144" width="896" height="52" rx="12" fill="#ffffff" stroke="#dce1ed" stroke-width="1.3" data-node-box="true"/>
<circle cx="62" cy="161" r="6" fill="none" stroke="#58657c" stroke-width="1.8"/>
<path d="M51,184 L51,181 L55,174 L69,174 L73,181 L73,184" fill="none" stroke="#58657c" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="94" y="177" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Investigate the payments outage</text>
<rect x="720" y="156" width="192" height="28" rx="7" fill="#ede9fe" stroke="none" stroke-width="1.3"/>
<text x="816.0" y="175" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#5b21b6" text-anchor="middle">A2A via gateway</text>
</g>
<rect x="674" y="366" width="270" height="180" rx="16" fill="#edf4ff" stroke="#9dbdf6" stroke-width="1.3"/>
<rect x="685" y="350" width="247" height="30" rx="5" fill="#f6f7fc" stroke="none" stroke-width="1.3"/>
<circle cx="700" cy="365" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="700" y="370" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">03</text>
<text x="722" y="371" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#1d4ed8" text-anchor="start" letter-spacing="0.3">WORKLOAD IDENTITY</text>
<path d="M284,302 L340,302" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="312" y="290" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="middle">calls</text>
<path d="M612,286 L680,286" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="272" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">LLM</text>
<path d="M612,452 L680,452" fill="none" stroke="#776b9f" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="646" y="438" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#58657c" text-anchor="middle">MCP</text>
<path d="M156,380 L156,418" fill="none" stroke="#776b9f" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#defence-arrow)" data-connector="true"/>
<text x="176" y="408" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#58657c" text-anchor="start">outside check</text>
<path d="M284,354 L314,354 L314,580 L638,580" fill="none" stroke="#be3455" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 6" data-connector="true"/>
<text x="362" y="567" font-family="Arial, sans-serif" font-size="18" font-weight="700" fill="#be3455" text-anchor="start">Direct MCP call</text>
<circle cx="653" cy="580" r="12" fill="#fff1f2" stroke="#be3455" stroke-width="2"/>
<path d="M648,575 L658,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M658,575 L648,585" fill="none" stroke="#be3455" stroke-width="2.3" stroke-linecap="round" stroke-linejoin="round"/>
<text x="679" y="586" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#be3455" text-anchor="start" letter-spacing="0.5">REFUSED BY ISTIO</text>
<g data-node="agent">
<rect x="32" y="226" width="250" height="152" rx="12" fill="#ffffff" stroke="#dcd5ee" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="50" y="246" width="42" height="42" rx="10" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<rect x="57" y="257" width="28" height="22" rx="6" fill="none" stroke="#7c3aed" stroke-width="1.3"/>
<path d="M71,250 L71,257" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="71" cy="250" r="2.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="65" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<circle cx="77" cy="266" r="2" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<path d="M65,273 L77,273" fill="none" stroke="#7c3aed" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<text x="105" y="275" font-family="Arial, sans-serif" font-size="29" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="52" y="314" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#172033" text-anchor="start">Incident-response agent</text>
<rect x="52" y="332" width="171" height="28" rx="7" fill="#f0eaff" stroke="none" stroke-width="1.3"/>
<text x="137.5" y="351" font-family="Arial, sans-serif" font-size="16" font-weight="700" fill="#6d28d9" text-anchor="middle">Declarative Agent</text>
</g>
<g data-node="gateway">
<rect x="352" y="226" width="258" height="316" rx="12" fill="url(#defence-gateway-fill)" stroke="#5b4691" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M388,246 L404,252 L404,264 L400,273 L388,283 L376,273 L372,264 L372,252 L388,246" fill="none" stroke="#c4b5fd" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M382,263 L387,268 L395,257" fill="none" stroke="#ddd6fe" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/>
<text x="418" y="270" font-family="Arial, sans-serif" font-size="25" font-weight="700" fill="#ffffff" text-anchor="start">agentgateway</text>
<text x="374" y="294" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#c4b5fd" text-anchor="start">Policy enforcement</text>
<rect x="370" y="310" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="328" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="333" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">04</text>
<text x="414" y="335" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Caller identity</text>
<rect x="370" y="353" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="371" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="376" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">05</text>
<text x="414" y="378" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Tool permissions</text>
<rect x="370" y="396" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="414" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="419" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">06</text>
<text x="414" y="421" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Personal data</text>
<rect x="370" y="439" width="222" height="36" rx="8" fill="#3f3269" stroke="none" stroke-width="1.3"/>
<circle cx="391" cy="457" r="13" fill="#5d478d" stroke="none" stroke-width="1.5"/>
<text x="391" y="462" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#f0eaff" text-anchor="middle">07</text>
<text x="414" y="464" font-family="Arial, sans-serif" font-size="20" font-weight="500" fill="#ffffff" text-anchor="start">Request rate</text>
<path d="M374,491 L589,491" fill="none" stroke="#5a4b7c" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<circle cx="391" cy="514" r="13" fill="#3e355f" stroke="none" stroke-width="1.5"/>
<text x="391" y="519" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#d5cce9" text-anchor="middle">08</text>
<text x="414" y="521" font-family="Arial, sans-serif" font-size="18" font-weight="400" fill="#ded7ef" text-anchor="start">Logs + metrics</text>
</g>
<g data-node="claude">
<rect x="692" y="226" width="236" height="120" rx="12" fill="#fffaf5" stroke="#ead9cb" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<path d="M729.0,261.0 L740.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,263.5 L737.856406460551,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,265.3301270189222 L732.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,266.0 L724.0,277.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,265.3301270189222 L716.0,274.856406460551" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,263.5 L710.143593539449,269.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.0,261.0 L708.0,261.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M719.6698729810778,258.5 L710.143593539449,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M721.5,256.6698729810778 L716.0,247.143593539449" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M724.0,256.0 L724.0,245.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M726.5,256.6698729810778 L732.0,247.14359353944897" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M728.3301270189222,258.5 L737.856406460551,253.0" fill="none" stroke="#c56b45" stroke-width="2.6" stroke-linecap="round" stroke-linejoin="round"/>
<text x="754" y="270" font-family="Arial, sans-serif" font-size="27" font-weight="700" fill="#643923" text-anchor="start">Claude</text>
<text x="712" y="310" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#172033" text-anchor="start">Haiku 4.5</text>
<text x="907" y="333" font-family="Arial, sans-serif" font-size="15" font-weight="400" fill="#946e56" text-anchor="end">Anthropic</text>
</g>
<g data-node="tools">
<rect x="692" y="386" width="236" height="142" rx="12" fill="#ffffff" stroke="#c9d7ed" stroke-width="1.3" filter="url(#defence-shadow)" data-node-box="true"/>
<rect x="712" y="405" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="409" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="416" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="420" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<rect x="712" y="427" width="30" height="8" rx="2" fill="#ede9fe" stroke="#a78bfa" stroke-width="1.3"/>
<circle cx="718" cy="431" r="1.4" fill="#7c3aed" stroke="none" stroke-width="1.5"/>
<text x="754" y="429" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#362164" text-anchor="start">kagent</text>
<text x="712" y="467" font-family="Arial, sans-serif" font-size="23" font-weight="400" fill="#172033" text-anchor="start">Incident tools</text>
<text x="712" y="503" font-family="Menlo, Consolas, monospace" font-size="17" font-weight="400" fill="#58657c" text-anchor="start">MCPServer</text>
</g>
<g data-node="egress">
<rect x="32" y="430" width="250" height="112" rx="12" fill="#eff6ff" stroke="#c4d7f5" stroke-width="1.3" data-node-box="true"/>
<path d="M53,469 L65,445 L65,469 L53,469" fill="#60a5fa" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M69,450 L78,469 L69,469 L69,450" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M50,475 L80,475 L75,480 L55,480 L50,475" fill="#2563eb" stroke="#2563eb" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="96" y="471" font-family="Arial, sans-serif" font-size="23" font-weight="700" fill="#1e40af" text-anchor="start">Istio egress</text>
<circle cx="253" cy="449" r="13" fill="#dbeafe" stroke="none" stroke-width="1.5"/>
<text x="253" y="454" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#2563eb" text-anchor="middle">02</text>
<text x="52" y="515" font-family="Arial, sans-serif" font-size="19" font-weight="400" fill="#365a8c" text-anchor="start">example.com is refused</text>
</g>
<text x="32" y="631" font-family="Arial, sans-serif" font-size="26" font-weight="700" fill="#172033" text-anchor="start">Ask the agent to close every incident</text>
<text x="32" y="658" font-family="Arial, sans-serif" font-size="20" font-weight="400" fill="#58657c" text-anchor="start">Closing the record does not resolve the fault.</text>
<g data-node="before">
<rect x="32" y="680" width="436" height="166" rx="12" fill="#fff5f6" stroke="#f0bdc9" stroke-width="1.3" data-node-box="true"/>
<text x="52" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#a82e4b" text-anchor="start" letter-spacing="0.8">WITHOUT A TOOL POLICY</text>
<text x="52" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#172033" text-anchor="start">0</text>
<text x="85" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#58657c" text-anchor="start">open</text>
<path d="M216,727 L216,768" fill="none" stroke="#eccbd3" stroke-width="1" stroke-linecap="round" stroke-linejoin="round"/>
<text x="244" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#be3455" text-anchor="start">3</text>
<text x="279" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#be3455" text-anchor="start">closed</text>
<text x="52" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#9f2946" text-anchor="start">3 faults still unresolved</text>
<text x="52" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#9f6270" text-anchor="start">Bulk close succeeded</text>
</g>
<g data-node="after">
<rect x="492" y="680" width="436" height="166" rx="12" fill="#effbf5" stroke="#a6dac1" stroke-width="1.3" data-node-box="true"/>
<text x="512" y="708" font-family="Arial, sans-serif" font-size="14" font-weight="700" fill="#087f5b" text-anchor="start" letter-spacing="0.8">WITH A READ-ONLY POLICY</text>
<text x="512" y="760" font-family="Arial, sans-serif" font-size="44" font-weight="700" fill="#087f5b" text-anchor="start">3</text>
<text x="547" y="755" font-family="Arial, sans-serif" font-size="22" font-weight="400" fill="#087f5b" text-anchor="start">open</text>
<rect x="741" y="731" width="166" height="28" rx="7" fill="#d0f0df" stroke="none" stroke-width="1.3"/>
<text x="824.0" y="750" font-family="Arial, sans-serif" font-size="15" font-weight="700" fill="#086b4e" text-anchor="middle">WRITE REFUSED</text>
<text x="512" y="804" font-family="Arial, sans-serif" font-size="22" font-weight="700" fill="#086b4e" text-anchor="start">Read-only tools still work</text>
<text x="512" y="831" font-family="Arial, sans-serif" font-size="17" font-weight="400" fill="#46735f" text-anchor="start">Bulk close refused</text>
</g>
<text x="32" y="881" font-family="Arial, sans-serif" font-size="16" font-weight="400" fill="#58657c" text-anchor="start">Illustrated chapter-5 outcome. Live incident records and evidence are in the App tab.</text>
</svg>

### Before adding caller labels

Read the gateway's own metrics.

**Expected:** 401, 403 and 429 outcomes are already recorded.

In [ ]:
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

### Add caller labels

Attach a metric attribute based on the validated JWT subject.

**Expected:** identity-metrics is Accepted and Attached.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
# These are bounded demonstration identities, not a production label-cardinality plan.
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: identity-metrics, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: Gateway, name: dd-gateway}
  frontend:
    metrics:
      attributes:
        add:
        - {name: caller, expression: jwt.sub}
YAML
dd_wait_policy identity-metrics

### With caller labels

Generate new rate refusals and read the identity-labelled metrics.

**Expected:** The 429 samples name the Alice subject. Bob remains allowed.

In [ ]:
dd_burst limited 6
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

## Reset

Remove only this lab's kagent resources, gateways and credentials. The console checks that they are gone.

In [ ]:
kubectl --context kind-mesh1 delete namespace dd-agents dd-tools dd-gateway dd-models --ignore-not-found --wait=true --timeout=180s
kubectl --context kind-mesh1 -n agentgateway-system delete service dd-rate-limiter --ignore-not-found
rm -f "$DD_STATE/private.pem" "$DD_STATE/07-caller-identity.json" "$DD_STATE/last-agent-task.json"